# Phase 14: CIFAR-10 với TickNet-L Large v2 (14 khối)

Thử mô hình 14 khối của Phase 13 trên CIFAR-10, phân bổ stage `1–2–5–5–1`.
So với mốc CIFAR-10 Phase 5, thí nghiệm này đổi kiến trúc, giữ SGD
lr=0,175, weight decay=0,0001, Nesterov 0,9, RandomCrop + flip + Cutout 16,
batch 128, seed 42, split 45.000 train / 5.000 validation, cosine LR 200 epochs.
Không dùng RandAugment, CutMix hay EMA. Phase 5 lr=0,175 đạt Best Val
95,98%, Test Top-1 94,98%; chọn checkpoint bằng validation rồi chỉ đo test
một lần sau khi train đủ 200 epochs.

Model có 4.453.650 tham số, 0,788601088G FLOPs/ảnh 32×32 theo profiler
của dự án, dưới giới hạn 6M tham số / 1G FLOPs. Đây là kiến trúc khác nên
phải train mới từ epoch 1, không resume checkpoint Phase 5/13.

Notebook đã kèm mã nguồn Phase 14, nên chỉ cần chọn GPU và bật Internet.
Nếu gắn `TickNets_CIFAR10_L_LargeV2_14Blocks_Source.zip` bằng **Add Input**,
notebook sẽ ưu tiên ZIP đó. Notebook ưu tiên CIFAR-10 chính thức
đã gắn trong `/kaggle/input`; nếu không có thì tự tải, luôn kiểm tra MD5.
Nguồn mã gắn riêng có thể là ZIP hoặc thư mục Kaggle đã giải nén. Khi chia phiên,
dùng recovery ZIP riêng Phase 14 với cùng bản notebook, runtime và GPU.
Kết quả hoàn tất nằm trong
`phase14_cifar10_l_large_v2_14blocks_results.zip` ở `/kaggle/working`.


In [ ]:
from pathlib import Path

SOURCE_ZIP = None       # Exact path if several source copies are attached
SOURCE_ROOT = None      # Exact extracted project folder if needed
KAGGLE_INPUT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
CIFAR10_SOURCE_ROOT = KAGGLE_INPUT  # Optional verified attached CIFAR-10 fallback
EPOCHS_PER_SESSION = None  # Example: 25 epochs per session
RESUME_ROOT = None         # Folder containing the named Phase 14 run


In [ ]:
import hashlib, importlib.util, json, os, sys, zipfile
from pathlib import Path, PurePosixPath

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
SOURCE_PATHS = ['configs/final/cifar100_sgd_lr0175_wd0005_ra_cutmix.json', 'configs/final/cifar10_ticknet_l_large_v2_14blocks_sgd_lr0175.json', 'download_cifar.py', 'models/cifar_data.py', 'models/cifar_experiment.py', 'models/cifar_training.py', 'models/cifar_training_large_v2.py', 'models/common.py', 'models/datasets.py', 'models/model_profile.py', 'models/reproducibility.py', 'models/SE_Attention.py', 'models/TickNet.py', 'models/ticknet_l.py', 'models/ticknet_l_large_v2.py', 'scripts/aggregate_grid_search.py', 'scripts/download_cifar.py', 'scripts/run_kaggle_cifar10_large_v2.py', 'scripts/run_kaggle_cutmix.py', 'scripts/verify_cifar_run.py', 'train_cifar.py', 'train_cifar10_large_v2.py', 'train_cifar_cutmix.py', 'train_cifar_randaugment.py']
EMBEDDED_SOURCE_B85 = (
    'c-ri}ON=DhmLQZsx5T&x)YHw--An^kR=h6rEDsNNk6)%&`J0Is8CjKd{>X@`*P3`PD+_n?2z%-7=56L4pNft_LISZstr^YC4$y2yvtl!X1p<i`AjFQJMywDc'
    '!2+>i!E8WqKKE;O&D|ri^1avHQL0FHH@n|+&pr42@7q84XMXKpc*Os`=zsVB{r7+8H~*<ekMM5~o($6IC^^eFN69oAZw``Cl(k!}Fh3iH<E+)*+X*j+tyXI%'
    '%%X6xm`{?c=FjtVdQ<Z}f07R4vETLlvERVo!)PAm@f@GXhWJG~n<tax`#8f_&@?|C$5A#-rf1OfJcc)|CUn=}h9~iKzL?O%r}XGDPR`Ef;V>RV*Yp?%MQ`F+'
    'IyleaS*O*a2dB||a31E+A3SV#p3uXKXdI5RXfRLGDfdN>^Ee*D&+QKVf(fPz9RDa9=dtl99LLkM`8l??OOK~<K995X0-nsXMO?eBy|uvnxBrzt`Q$eM#Q3*g'
    '1?F&`4Bk&;XgwT9*;yQ3bi(%b={Oy{&y5wh3&1}ryxSiQ-U!1F0r&s*7yi+2{-^K_|NavE4b#i%IE{uOfu%XS{`8a0MV@V*CezJ$dg0Bk=jZ8ktLFRuK{~yN'
    'r%5~=#9o%BbMG>XX0tf+Mrr2dgDjcN^UYFwO{iH*CbKk~d--)<8)fN4mTuamJyC2DCDU3h8F^tijV5szdcB_Khao->L%$32^sudFe3i@td>Yj2g?awpFaMnv'
    'zy7a1dW3)Yln^3kd<qd8U}!M1XYlPM3;<>kO@{!oXqv-xCOJ*&;KlR(cN^^%y^}v$o<OFYrbhD#!_i{C$Z)7)4$*X)&LdpYye6j)jStfa*O*1~^Ko)2i@$}R'
    '{9=AR1I&}ppCyBN!+V+J@b8-$HW!T>-tl5Kj#abM#bkEvMY%Vf$w%`v8;JJazIZ7cdohX5V$lm0XfEc-7#0Dhqxw_uXwY|<rcvP{$zcSttOP^D09%bqH?w$_'
    'rNhM_IZejN{90B)qz>bYI0LMLSvL5=WtzQ@vs&#_?`RGnin5_Mg~p<B@;yP&1DcS-oB6U;snIte6|WBWUw0WqCnG>EfSmc!_j+E?YHmMmZ#Q7GZ9nOB=yQ8_'
    'r>@#KI)1jgiO%+33#)dvw|3}rx4m1h$u`y<$WCjo%>&up<AJoE)O924jbwL^N3gTYpL@G)ITC1L|M;j&m~@iQGl4xP)onJG^w-B8Z)gK5LfZ8L6ZQ?yfMUI2'
    'l{E_&O3UK6wc6`%UWErQ_m7Sat#K#QxwH3ntK3N&DK0cs8^$ArOmQesmdHwg|8zZQy6$cK7(bb?41oGi<015f1)A_YnFaOcIK7NB_yo)zM}s)<H~fa@`^JNi'
    'A5bGeyktAc)C=^XV~n!S{o#LE4Af&PtI*>tO7htIHX1MD!z@d)VB~)@eLn?)$q<v<&%424ev;jO;SJ8y6zY5?diczP%_4uaK7*0fYJ+i<=iULG9YK2f^LQ{9'
    '!vI?Lj_A|)8VGRy-h&ceI*qZBNt(r8{zr=lK4y4^#jq1`pGQaxfT(!$c$!1O;92}_mcZNJ_l~3MaSE%}frGi;tWh8P*;%e9Jq^#(F_aBpVe4J*^<r`gWH=qE'
    'F6OB>z(L^O$cv-FIrr97P4MX_TR1+4QNl9gZ*P`d0l}}Un&$AEa>hI?#E~#erpY`EgFGIO8ggu1L^{9(*rD8&da!G(9xOIcZ9pilRQD2~#c)1iQ2>gZ=a*g!'
    'e^OCytyoe_=S#1x8+YP@&npOc^GAy$i?NaEG(H1rc@bk{@mZ|<MKU={^Tc{7_=dXb2j|_CReG|r{aTE#Cu*C}8${!=f?hH?>xv`hm=J?@nEAj09u#j*hsnhI'
    'q~~oB#1JI#Fv&wAJIIR7QW$zrUlTC#)eKM@=I+S`yo0ZQD$vxJ^?Ad)g!y1+`FS*pPc}OJhWWGIS7m8e0AyxO*cN#ZVBLBHe^Vb@oqA1`#d9jetq&*TSsa+a'
    'EllB>n`GXAK5(Arb*s>ov(Tl{AvM+JVuLYItZQ$>TC~kg4-dMAS7`dWV^#Xxs<d7cS7yIgRh2Fal{!|XwfZei7+2)vy4}5o*H`WCRR_-pV8CnEnuUTm^#leY'
    '<qC;u3h;HkIZuNj?#dpnJAG>aR$n9o@r1tCf!mw`;KL{n;Ay=`yWyGl*z8XnK4);;FxRQMq<}Y$aTw7##K+iU{)iT5C~9=Q(=;8!*pCI{?|IDmSH1{~l&VqV'
    '?WekL9;CpI<*_3CitXZ)Wk%fFAkK5xH(ig9g@Y`e1zP~<Su_OfJL_!&?(?g!(=7QuozA22b6CuV_gbE}4RO*Cqf8@=fKbtdaq->9viXCdJb_aYSj9P>$p%kz'
    'qFd|N<f$vcrj}<u3dg`cL)8-tcMNb;O;CLeQF`e4Sv(p8Rq+K=Q`wRYkD$8vamv(iNf8Ch0HM%Rpy<J$QLAs#Q^o_L`xZ)gKlWNS@e#mjq8v9J27)gTgsP`b'
    'y`I%fy@dN^)Cxq)T(r8FUXvAIxf@9On%6m&731O2;uH`o5OR&?d-ypX7Lunz&;d-w2_2t!nv9b{GS{O(-h?G9;73>xtYRgw%`Q?p++taD4dN22Fy_Eqpj>x?'
    'Y%MakLm)EtGu2z~HJmucxk1E{#I_zaZ@&=DBfaBq_G{4-p5o9|Abu7Y0X@;49)>wZXg8V16Mf_luZZFhsbTrSJnxx%HyPe&96}vv-W=2kJTg#uaux!}WEq_4'
    'iL9tg@g(sOha=GhfG%4%n}m`i2-7)0D-wu~>gFaYfI%>7xDfUh;uh`FRR@GzdJ~YarvwlQbcsp3L`^6Xm{6*VqXmu(U~NVTh|oD8=sXnf`FL?)89j%SFQly@'
    '*js*TYN5J9=3_kzNa*q`L1mP&zy_+SuYI+vp2hPxo4{7c=LvF$*B%X0jtiSijVU0S0l$Ph55|%VS@IV%QU^#fqGbS?w*_pnIcVgXn=MnODhp6n`N;xKh0~bi'
    'GvF=SaGae%(>G?JTVHS-*%Vh<@Lx>j1B6Unh>QTvaGr*07n9VAz%?L)vg@Gho%D%29BA&)Sdsc|O|p-GxIllS(|C*}V<;4(Ix-OOv*i!OeJWc=CYoyrMj8Mm'
    'pL}U8{t8a|nD-UX{X7|hga^WvB#%s4Y48RwU4+7k9#mhRFfb_rQCjY>O!n}g?kkk6das~FU<3j)kB32V049hO(JPH|ph*d=GVuM25ghYnx`@LWq>HH}H5q&#'
    'q9CV=0{>?sPu-#?VkwMfzVyt>tCr&sg;)ms6c7*2r8(4(au(7FnkePq{=o;v9?GK$>{j2NiQHK6)tuHv0HdphXAn(msfsmJg>GN0?5Exl>iTeXh9(X0;K0au'
    'NNF~VvoabqaUp|xarMo|nw8Mi8fIZZ8&4O>Sg4gHdug!UA?mM*=l1@XbgzvyIC2MxWO2hyD5*iA-*zzTKRAESG+8h=T3ali^p!%_AkkuYgf%8)8z%k)sb~r)'
    'Bq_<`d~WN&@6X!njnOMn^ChIbl9HbgB^S6(D~*bF7!fXVpq3uBc6mC!h=W4O=H&e_$%0vw!P?|Kf<3T*fWKk-o_-1>ms1&xbHRiWG=wsfxNrj?J*}_w;&C3A'
    'Dzu76Gc;<bLCc~JZvc}`a|)WeUPEGBPkc56J_+W&VWU@p<)=Plg!;X@e2!K^N%#SM!kjoZ1Bd}G3vlpRpw^<^;8}9*hT2R$`Niar?XlIBJZK{oZP#vru-$1a'
    'j1fzRQdk<B?s8|yMlwjpxxr6<>Y?og(EXBz<6S26bCl#Pv4goN;+{&fO>S(N%aA#$A989WhZN5R<Wnv*J72|=su4;4ykX*d!&)ge4D~dJRa~vIRHvKAnKe8|'
    'cgA?{v|9J-Oh9BMj#>vAQEt(3L+upDkv1k9-G`oN+8(V=6E;fU>#)PsWK*8fn~OIynA#+sq}jFORQu(ek<!R%9h3Y}oRs2+IxK&J;P<p)a~vDR3YJOZd&`tv'
    'G{EtIBMw~y4Gjd#&OD>CJn@uTAJ1j8DZ?J%TMV}w!vIFr43mXoJ@u_&y_87SGk-L!%_PyOXYF)D?;v)((Qt_tsaM6rOiLlwp;mb8JlwdDfYMn!MJCly7k?$6'
    '2wp*zXilQ(f*=X92a_rT(7Z#T6^$f8nUv`P>ecP(uLg8jEa_=XFKoG<@m&|tJFlm|>epPJ^;M&3&#=bpGHSj(_0579uBMu-I@rX~i@4=fo#4x*ur5K@`y4ns'
    'tNv0D8zkITuC4&I3GC<6xk>-G65?w>eO<`g(6=C_3z=Nia-0);{mv*?fWj{p=hniDPTj&t>qs37Laz~~VFIh*HuU%Yy}$X_-v0KZNBH+I72Sr=sF)>4ZrFL~'
    '@FE$KHx#Lh=jjAw+$aO)CmNWpDxR*Fd%-l-!a6Bd;?W3AmG!%N6AkhU`Sm=?(V0m8Oj7v`{o&;IG%xreNuQj>sj&7a@1b}y8ztko+EZzz>rt)t?#-KHDR;qy'
    'pt&$K#IB|gu}<3k+Tq(b2VaNJUmU)Cc7zTs>^O(M61F2AJdkUe=la(?owfBZ)^_B+y^ZJ$XB6P7#EOZEL?%SFdhOle!Hc(t=5W}Vu%W#O8>Zu+YnaMSuwi)Q'
    'l}F_H(J-2Xr|~?Zfv8^{^(&sya4jc8_zh*kNiwCc>}BX`-S9VT6dVm3!ov^;s5deBr?9JGqv&ENwa&DZ`FYgY*$qg3V4^-u&SEmN31n=FB22PTl3`q+D$R5j'
    'PXj+Y_3MZ|=YWu7<8UCw-uYts9z}Z88U^EMaypE<q9mYEGz{9U&bIg1!w-;QPJO?iL*o%O7qcO%O{qOgha}#g$5;F<P<r)rkqzQ77Ta4o9pK1OE<YjE-joO@'
    '9UTE;kA)&KkR_OW`1R)5I6V#g$5^ahXN4oeiRAp78O&!)yc(b+)vY(9JVfU13OX-BR1;c)LpYDG<^c>eC99>sn2$D|n0`q=%YH`d4gc1gR1O}BQgw6s1os)j'
    'oThp!xS4AW)<HU%B)G_$Xfy}s@!)+3h%#HugA@NOA>7I0iw)S~Ibq({hx^ZbV5<k0!yZo0fvZ*j<#Qm?S<mS2+2OaZzj^r*src2LRWi~^i$&_IK|Gs#!J8xE'
    'n;Yg>52Eon9=_#Y^t$etoMK`GqTKibBX|M)u2?&m8O%21dhnFLNUyVyPXt2vhIg4E9vhe>X(R0i^;D3>b<|L_QBXRMk?=+tI+94r4fS*U9?Y_Mlw9?wX;MXf'
    '?)gn0VKTz!eseyV!B6O{XLQevr?9qTG}4ZQ5N@FYPbi={s390CZ>TLXP`z$=dbZ@>J-)WA7*K~sK{L&bX6=m)#TL_XGJS895I|>`hj{J@z+a#QB^t1oglUR5'
    '>%vB&+3-%ny3HXRY<WO+$u^6kMyKJyh|^0DZG=v$j^M=2wJtuFc4=YUw3pb^Y5mbNi<C!jtcn58RJ*`>07(kZ0iM8wiK^n4X!aCPftbc5Z63isnx4IQ6HMbv'
    '0C3!s{*1iZXg<j=nrPhmDK9@)Zoq2B;~^qg-ZM{gvogc@JSMKR{31&)^FWSl{ng?DG|%EW%{_pVpa|rKXUs+dMvAF`hG%g$%Mj$lqGQ0<u+2#u$IgVxa!(8&'
    '@U^FKWS%E8rKd}#7wLOs?3lwcoff1LVEHj=A6clTLlco4m!jTvr96jLqPfuKkwi~C3ZAMk!tzToe)FUx3pgEX(;1vPY!DYUNV~-@ZpAlB&&8foPy!&hyIr>&'
    'QOW}l%E01L&4Z9eLPvMX(o^V8ko>W;Gxo!(m@U6-4uO*zoC7~Pm@VL6b_?T^P`Hi3ai}KE4fY+XTUOT`!{CIrMi^U!u}-RVHzfwC_E}JX1L`6Mx&p$)1;`4c'
    'WI)`$*nL=nMvjBwDHNu}zI0JONT>7UY>_Sm{ODgzl*{<VCD_}s!ELb%9G<Bqj4QXn(%nT8Uy8n(g9vyD|0jOEUN}Pqhu!r|oJKxWo~mVr?AF+=njaI&PqD1B'
    '16Fam_BEGp$}SNzg`{k#CfF{cj((FBWm|@<GWEU(UL#=cPw>^kR7O$=4BW5F;<=i_Aexa(mrc(qCg5|d4FE#;86cv}5y*As(q|Z7q}@Q#`Xb+3WA^Dp@a4&X'
    '<)GJ!vV+AiQbqBHT|R^(FCdT^C6YXhE}~={osOYFhTKWN(6Ex>ami1_8ffL2G5U%Hfmtp)SBovW7n2rHFP(sVU#<$sqfnJsg+`+nPAm??I+sg9|DMRD5=L^p'
    'Cb^oGD9Fnu%88TOS`Sf<m#nV{yLe2i)3=x;@&+>BlFNs&I_$5?M)@@yroVZ%FT7nn_DdL?qhrU=%M&C===Y)762ptY=rJ#0j6Fa_#H0d7EgDymn@WJbA!l!('
    'P~Y}<DK${(o@JueYZ1@jLY;C|H@u99F<pO>kgu#aTx4|o$N6GndR(~h)xed)Y$T5YM^eawe;$w|>7;n>gA6$LX)F#hreFFvqqds7Sa7TYk~GnmFW2;z87z=z'
    'fMO$I3s}5^(y(}w7`ViN+#tmCk(z`fHz@{F>m{1NFp`P{PKD--M8i|mz5&q`KS>!U+7y-TWuS%^7<m>>A}Dqx+oYNx;ouxlHAc@F=%yS%B8r|s9R+heH96iJ'
    ')v1q=S>zNe%Ed^C)tew>{Zv;%;-vh}AcnqF32O0CtemMyaSW2YB-8@p!V#=*x#?SpCB!dWhpHErP0hA=q4WbOdKyJ)LI9@OqMtQML5Q+>V3c!#uD(nQQMYj9'
    '=2rwqZdHtyoG~6pfn0bZsCw;@VxZBtP`J3PfI_!*s!G#cX32Xc$b|-_Cz%_A<Ku11kt`fXrbcj5>b)<waJT4mW-lKh6$RELbaq2OCK{H@;)LWFR-7dxe_xDB'
    '5kykt(QzWM{?@nHLN%j;9{zv$e_sFo|FZY!5&r#7Q4cTm=RymQIJ9w!-gu}4$>KqZ0j32{(v3VGGaS(hzhKWMrJ}#9?%mX@Pf#H)zR$+doIIshYR*|rZvNol'
    '+9jIxw=Z7G@LFxAm5m#55e?IGs8Ni$wOY`Y3E56yslOX##YDrTtqqf{U8Myvj<bf%(KiZuiiU}ktC}@fxJFTR+o+(9jnWBM7jRLD1|&fC-yMAY;`s33_?vf!'
    ';k(0cUmU%7^IEF-SodjYZA>`nK|Ge4BQZSQ5<-W>n~H<1wL)2TfBbI$#p}a&x@RWTNJUW_qq<|7#4nf>X&Of)m)=tcaTaDVV;hsYVjs&$*MaKM6ODPYG326v'
    'WdxAIZUM>Q3tAe0JZt}vk;J0$rF8a=GerzU3qW?o0&JW{c{13D7Qkm`esLabk*Sko9TgOAb?(J-03w172M91=+_j(~f2|1xvn4$dLG*m*D%jd&rmzB6P6{vk'
    '?AZZi+<`#LMtDR8Jrv$oG4ukL#TH~GHd--Eo=;cM+A(ol<FZqmtUgEjn+?C9&uKo-7H}F2W7O-!1TASEjYe}4$>85?K~6YIM`ki_Mc%_?AM~J3B<A;}4Zu=+'
    '+KpIDhuMZGE*m*zMV(oK*b)PHedXALWlK_o192pDjb>4E1xxHatOUKZxmttX*fX=aCj6!K!0})Kl17xB0r6vBA0=&xpYjWBsD_s<rp(MX<dKP*<t!!AY>Ff6'
    'k}lc4GM$DPPMO1OhKn)ge~?~4#%V<ZZfZY3EkUEza(<4iXU>JhkGk+df0pE^o&Y=?E(QQe-~k7`H_5p5MZq^c8lR<bB%EXBk0=?dkgxGIF7INRFJ?0`?lp~Z'
    'oJNE9fY>=p!N3Si7UL#w)cI%&16+y3SJaqJtI7&-TuSx)GazZY+ah8}GnU1G2m+l<efG|-QqwFsMu~R%a6T_-o5~okq)V`Hf#{X3rhHUs$w*kes125oJ<@B!'
    'Pl;fKDg}q4LR*IA&aA-zK;&kjs!+@DK>$%Un!z?;U2Bg3(2!d$OsC^(>9{OATJykP0%{i@fKH5o1qb#h!6#gvJV&ic+5})#G-Jn*4wEeBqLa0vf|K)97#<Gd'
    'j4i2C;0Pl<JChPW^u5nLX<7{Bwh<NxyR4EY<k(Kgp?wXl%g8^}TszAV-Y`MSk4QvOsik6?QXEC2Fa+T^RPBGyh%e$N81u<(2W|G!!9(J<qHnZ&FsYPYPE}St'
    'k**>boXgD6VhZ(R5|^bS7d7>a*I@{>k>uPEwZewvU>4&icHttFO^vcq5E`8&h;;A<Bo6CUsR>JYEXon>K}2CX(L`ecM?hd<t(h&iCKQ)g0j&YLdSy{uyt4Az'
    'co_&{wPHN9^=}0)I!I;&{rB;8w}8R$h&3|s1viW(ESdFBwA&9xtd)_)+PzE-@3(hhB*IKg0fp3iSr;IBlMNFeN|X7d$CcOGgzuPY3lM3+9$LEft!Ou^BVVAj'
    ';IH||D!sJqi%V~Q_4XUt;TLc?qmDsI`fOrD5|Br_ieC!9L&daMIOD`HmJ>!Qwb)7tWl$A(Bw57zYQ4Jd0r*0x4^d<0PU@u`GDV4w)rs{|P8s;x#&{MJKZNQL'
    'AM>F=jP4x<N)@mtGw74C4y2o*c3eDN3}JNk@X;>e=B9DgUE_p#oMieZE<|Q8j?qe=0$VX+or7lE4QHF^Bz%{(5$$W)l~jJ##G>hSfbd2|5AakY^jwVt3W*4^'
    '^zsCdr(b8X06!}yP~2b^-YxTMta4kly$<g+Ycv-_R}?R>!4s28Z96p(OCvpi!IgWFi8l6xlLmmH>k(}9fT6^OdP39b8%AZ?YUsi;Tub5-D?`fyM;&E{1C~>_'
    'f-X&iIjJ}GLu*GNI?0KdC|RO|#rjgn(l+0l+qzJ%;6w{8YXE#;;{-ED;{uj<SbY*!1)Gb@9g!k&2m;et76a=;>KY>eR1rNZ$%wH?N`?!yQOWO%VMn<%ic}GT'
    'I}Ba*B<W!^H<S;g-sP^W4NNj5EJ8?1hft9@)6`s1>dX5K_y}4N#1zHOco@s$>B1Oc*($>V3p0`0Hlqa=H5sE;EQLn<Ki!fP4a&~fwg^!rX>drn>QFl8l`<_n'
    'PsZ^fy|YwM8=@6M?XI$xMZmnr`MGEvy$}Jf0n@%fu;p=vfla7hL#Qduk#9u~m&GQ>P_+`x$!$4@Kl0!3sJIc8T~Cf#4HjvPXwtjUo!|O(2aiW1DQIn^*{g1k'
    'x6&-5E8u=Iiy0j)ON8M+0|JmtRc=i_0wN(%#pD8D>Ib#;I#+>WG5}EtMjR97nSxToj4+`WRRK4J|Ey>+O!ZL|a}8~TJDAMnNsRGjN}bfO;#gQVs|x2S{Aa|s'
    'I7JE-SWU`<Wn(V}skw|MR@|CVO9rp$AUOuT_*E~3qE&{#U?w15O*3RvhP9X7uFC3|vg8tq>_jf{h0g`6ZG&D*Tcd2@hCQm3XIFm05Bj$ikd@_J;)pPrX-^R3'
    '?W~t98L?c%EE>R#60n%#k^<JsQ&)NcN(WL(hXS8yQl>a=TQ_frC4($BT>vY|U0_7QnJzBCjq5ks_YJUgX_6fOTO;{SE=l!Jgk<P*m{<@V9_MI7Z4n+}h7Q?G'
    'Y}2!1y>Y`3RS?!2*fOX&Hay$C;`uE)C>VSMF{<v#XxDUdxWc5wJzJg4J~j;xbHEKU;HsGKGzKLWK{s*YBd-Ko3?$M?I8J~E$xZgaVfv?OV2lFFtXR$CYF@DW'
    '%7b$`nvip7`z!DH%QtUxnae`Lmwk6IW1*pit=0l^aZ0gm`1_c<l1f;7o5ee=bp9x_14O}BI;B{}2NCjhOWjV&E+A_-o|yb9bz!;cC>vYPaf~Xq$)On$z6;di'
    'O5WAt8VoL|%=Bpg8^cVN!3i>%Eas@hH^`*H<*XSiB5dW0F)GRB>?F<`$AQd!+3-vQpqZnp(43C2vYnuM4$j#W+6B0K)u69B83i@#*~|t_rKgESRc)Fp9Y10K'
    '#gpYkx+M22n-GeS(6ZL{T&G>hqFdFbMvX$NmA2ukmMr!?j0PAF1-y?oGpsNwEVscre6a)~lv3XDV<G5Bt$4!_;(E(Xf)YD%(1LnXs0h0Xu-11vZ5|-8KAg_8'
    'bawp`zDm=R^paprE-J@4oQ4vnX{~_>nn8(4_%gFPwX9vehy=&093~4ZAPxwq)g;e{9FGUg3@-Az3H!C!Fsw&yc#j`zSCOKJMz3T=bu7Bmo1s_jO0ncXb$U>^'
    '`V?B$p7?Btn<AvOpeJ3Sn)Y+Fx>X(4j$TN|g`_W}qY+KEdCxq5T?LTLjK=9K4`Be8Q8u({&>Q#WIa^r^Hb6Um+VIN9ejzi1akimTHPVU!-Jd}+joGr}y?b=P'
    'G>jL>r=~NJN6iL><2^#cVzd>lb9G={y3o7vF|#U$YoOBA%)4tD!%kZL`Wo?Xem)<rZmZp|3&$m?p;Xxbg*S$>2h;u%N}8V6UTRPIFfK?Dr==}u66S725w)eQ'
    'xHW0kiL2rj%zD~x(bo|SMr3QrU(8`m(jZ^1zh2ufro2oQ8haxF*1o|70cjz<ykvjSsR*8No<k(GF&t^{MZlCubf3iwwB9E3W|gv1_S>_pBre6wM`Csr6ZVmq'
    'T@_D6A-16qpSf4VwOXo}yR@JurRROja-c>ftjornZi@=cbs5pBUAoG<NxO#|-e}r2V_p9>NATX-4rlbMF6d7>Q+Vyx9V_n^!Z796E!5BIZUDVQ8fL3#$<jj{'
    'BeSF_5gm5nS(YwlIZ$9Y2Qy>SF%yCElyc%yjJ~B2QtXt$^vgc1Tsw#q?gmYzQmHvhlr(+h5H^8o_rzBwBIB7bY<NHRf(i^Z>Q$C-nnwf)yYXcthX8;19Oyct'
    'a2gUXDCQr(xDP<bR=aupSj|wzKgjNHZykq4K96Sxn<=QVjFc@;pm=rJIJ2{+bR}{~$W))*rWw+xCP7}vw_=o#kCwHqELUBxwB)9p(|RqsUf!x6kWo_&>9+3n'
    '8oMtlRE)*V4s?wjY=&Q95UlIq?^=jpl&|Wwh?Zp^2B!9gCqq}_RAr-@x9B>fSZ|HvoBdn8QIY|zhQl0vDS>W`)3Z(5b4;W$H!7yDGkk<m#fXkvg2*?2DW?Ky'
    '_bFs}#9<}g4PisKwKcrmyfK|B@QL^GUGHX`b(_0qx0oa{0j61Qg3l0FKkIj!ozd-&P$Sg4(dcNs^ifz@<mYB~*$S5BON(QX$4mY`nb9e~&Ve?|(z6Uxp~$c|'
    'Bid~i0q->Q*Xy1Aro?nBGGM3}s$r<^LUbyHL}CJ6%IIlr+FEf0+?Lc51<r_^(SAt2s9NOU7ORAfA5c5DLNmwXSW_og?PKmFL~APN?xKdP*FH=owKSa7EO@_t'
    'V?~LUF@7m7FU9=3W4WYW2XYv|a+nKoD}JhT=$044J5FaC?cNOoV=009g^Cz<RNx=~gZUqHo;-Slf1gxE1*#ijIXv)1o{ouk<y?F+XbBnd<7Dvubv)mA>Al1%'
    '-bKgc33N)1X03)-%cWUzhSN0ypC)8EhlA_;n4<|v+(G$Nx2>DZo3)2H!-qISJI+v{>qEfdL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`uf'
    'z~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`ufz~Mu{;X}aTL%`uf'
    'z~Mu{;X}aTL%`ufz~Mu{;X}aT`T>Xk`+xPtKiv9PA3egq-!K9Wfw+StHM=(B>YCGOt@i1spVmJ8(T_Yd4_}B&1{YJldXiq&XxDg0mq~seD23vy6wh42Gh+CJ'
    '&<F*2JRUW?E7O3+kht<5Ykj2b0yasWoK4bX7+f`g;h&A8K`hBUcl`B7>p6?Zi{0P~#E7kCtI#_j^lr1I#~=pz0i(F@2(WQPf4x9~{wlgbc~P$)DlD;Ozj_Ss'
    ';+Nk9Hn`Qb^p2KxS2VS|W?MNfWw|Y0;p%GV74^PeRZ-y9lF`eBU2pv7l?^HkaS6|C^Lm}mUm(drXZCm~j9>e@gPU7+U)5#&CaQ;{xw+tVy=5m!KEP805YgJI'
    '&&Lt)anldlGEIy^46d4Jf(}~1es8qZjyyX2W}3{GG#`fOCmDv^e)#}IQ_2*L{N-2qA1&he`#3}`QJVGIyGzPwsPq`$9cAOI0Um5-QMrc3tn<Yz&VqVV4ct&l'
    'iq7PiNku2VZf+LFTQeO(N6lH9j!i4f{t#q?<Rad`IC~4PI*QdMLw~z&*G6S`8@t2Lt9F3FW1{Dtf*|rZ`f+iv4evCG@`8nv8rJ~9E(-Le3}DC5Iu%1JkHH$^'
    'U53^v53RJGLL69*)wpVEMS`n(`321+$Q^GCNIBmVELPOC-W#V<F1OSH{);NwZ0&9389=w$+x75Tj@>d7;#`J7X&agAYj#^IXf5?p!`1sZn<9%(m|uQvAjcZ('
    'du1+$xxV<qP0Na_p}Qeav6~5XW%Q!}g=Lyo5j32IY!l*Dp#d~m0n+f!P~L4fTL#T0h>xS|I0Kd&Ra6;MZtU2xo9xYl%FiqrBET3xW_Wz@WOJ?MTI*=7q1AnD'
    '%>@*r<PZDREE<wtzGXkA+2B8Q4T`O+tv`jJ*t#<)+G~L#99{<H$=`FrYs~f+FjLmeSM+2os)obP)y}UT9Cq#uht7TBU{R6NaXNT!QxS2Pz}uhT%MW1DmQs1I'
    'dMgy6<6_?&kTJpIcr#4K%)B{H3JE^fD-Ary%S9fCr_(B@gc5UkhhvpnhNFkSlwX=+g=cX0X2eCk_KQ5Lm#?Ua?n@VU3G$m8pmL?vLd!4ZB4WAuqk?f@@@W{>'
    'jAiK-0-xc<spTv66ZF7;4Tm~F7g?B59HBzrhLO8<y^@YztVOSu=s$?xSvFNw>2z9ErE2_FWtP@#Li1229}^OR3YJrnRis28^Dbwk>YcxQ>#x}Rm78BoR1?_R'
    'J+|Ju`)(PBylkIUuvkksj@mVa=K@arQFvzn$wFU~Y5C|V@Mmk8x9+jb?O(uB?p^IYEalyna_=5Xx%CU$mBUM3&HeE>ShiX#7HsvZxE7|OTopQ(SwQ#uyRY{;'
    'i~W)ItegK?DhI4&jqZUIwpjJ287F>etH1Mut^TU@U5*_eu=*xN_{A^%pQ?S^{@82c+>(Ee7Ue$Msq>K+#Wt}2jlVU${<XjS=n?+?rZ%uk&kSv1PvN_f0Cyf`'
    '=rK}rWVUUpc-$rLnxk|)jI*Uh7V~7B*OZxV9!)`n96n3W^1xaCTly^maCO#7`jlb0+!i|JVtNL%O)rSvN8XD`bQZsk=lG;9JzJ=S;Cw!xbvHLfueqq&jE9TO'
    'Xqe2eqbIwYX7$ZF`?S$N$LC2dI)^WBua9v8o5nOeokrEF7JJ`eoc(tgHt0<j`CP{Db9&Y2k^&iX@4E!M&YQ3>-;rzkcNlrWzZows&y&G9g%_$EfjX7iH}770'
    '(P?@SH?hxa^4A02HASaA8DTC$A2B=N1V%cD#^Y=6^cu6RQOf1-RA4%8&hMJm0@QSz8d{X}-n+4o)?Z{}^su=zu=Vr{$FDyb?qG}U)6@2-6YV_RjpDu5&fe(h'
    ')8XE5Z|BKwtJUhnqhd7#tH?IyS2rGC%`p{0JPc)dbm<9v8e8r+42P-P#t%JbwZausj)$kGPe<+DCp+8mV0U+?wR^fVYCS#Od9oF^c6WA{O}Vn|su`058P>wI'
    'n3U6qeAvtf?365@ltEM1nBp)Ha=Hm>M1&f^_i2tk1$mEM`qTsYn_aDnj}@M>cqYPUdQ0e36UGxQ#?mKmner5e*;8Yc<I`WwQL1l_p%xz$l(L0P4(g>{ilBws'
    'K>RG1!5n!477h)cd{qc~Ea-oy$sa9ZHjGjNK4mYa@RjoGqIQ|~mc}KkcHv#IDuW2C%onq9%qB|&8h4jWhw)X2B(Bg<VE^PyUiLGL)^H84ZUgbJp-*5kn8fq*'
    'bf^%Q#;*1Qj{yB%REHYal04#n?=YJ!29859ikoBd9C^51_-RCE*Ce9EM|1|jX(Dj7X*49#?Hmn~x7ST)XX4<^2g&TZnWQ>$E<%4)`Iykm8hTMndP1?XCoRtA'
    'rUGZ7&WR=oe*dJ~@LK)8y}_$6q+q5thy4=9!}*`+EM=5Yr<jHuM~2Y|;h6Hc6CR$<6A~e`33aMqRE?TTmbWK(rRLAmWXed%^q@c^jVP|-_L&=~?nchS*zE1A'
    'o0DVKy8@rCu_=0mJyOwr-UPNa(7#LgA$BHZQZ~xv^}<e)et|0nDSSojvAI#@S)3&oaRFI0uAIS30Y*=@<VFKA6S=CVu-RNxq)GW*DE1q+L#<+F)kJB<)IR2#'
    'iWycA^d^nDw8-{0E|Tgcdl=~)?MfT?#y%Dl?Xz5vLF90o-nUVPL7$iBKy8f^?>l|ceTU5Tcf@^Su*7$CzJFJ-xkttkX>T9i)^fA;N}P*rfTxL5OgDxlcYvF$'
    'nQ69HHEnT8OIm)i^Q86BD3diTV{4ZC;r(1m;UwSdt#`zhPFI6A=P{g5nANCQ2z^%#*wl(>Tqs{;Dq-7TE(gQ$46wTrRhfvP7)Is3+nAqw99!n4BNJI0LcmJY'
    'p`tOAry!ZU`gco?8@ZIs)&(ve<b+1jue(s0c}3Ik-Yw&l>_e63@fpzPfvR%MXNi8x-4;02>fT+=>Ux0yPnk~@3tIqVabW>GVo{f_CTC`xE+(hIKf@Zba984{'
    'eqeGVrhxcd%~z_)DAs$)Y@UfS#OnyDjHeb4K*-J`OghY$g0PApna)$z8o*wyX+^C_N8}zi?px7{WQm0oZ!hg=F&>+Qt%^c$<r6Jy_A8_d4lk5|u+gYeTcl)`'
    '5M=5A{(7=Ml(G9PMD~o~D9TtKSHlA_T0KXRx?jFhp)g#r3^hlJ7|ca2fg-XBRKch$VvB6U<rlOlVu&uuyzj_n#UVrfAs0u@&1W~ap8-_qQ+esTAfg6YFN1>b'
    '{9(OdGo1mVTQ~uSLxz_G<XEQb)ZD_0rkDgm3t^+z%I>gyvKlkP%$_`_OJwoKcr0FP;K4NBC$m|sb`3@*RnHoxYU@nD{q2AE-~Rfad-MqZep~BIImk~WnB%+8'
    '4(RM=*QU*v$uy2K@74YRD<Cl_-pCFn++(~MoP3;aMqNn);uye_Zp*Q56=Qx0ze;fqL!fF0Z@zhb{NnXj;lZ2N-yXu(H?NV?Xk*v#rNepVz7XRg@*&2CH(-3h'
    'IXpd7V0-N^Xozxfe(mrN4_<!r?C@Fm=I!CTeQf$jY4d!v?Ru!j=2SEo7e-B4X=d=TR!37QbYllJ6q+U<B-doPfL^Vpjt*hbDU#$8!%y--#C1GMEa^?s3wjBS'
    '$SZK1x&D%=j8LFky3DkHrq3<F`Yl(drAQYdaU<>)b+?&JO+rclyi|tsJgTJWqj7YW%ggI#93(~ThP)h!`t>M(iBJgx%gf1p(rjZKU&LeY`8b^s9>p1^5^UCJ'
    '^*lI|Vpir(J(d1vgPBF0SExaxXA(y_rE?c|yx@I&)4?_0h9^jrhKtBL?**6^vD4XJnmp2|M#|~a*xTwL5TK$+UZ>JD0isMXn+ZqC>g-K{ASQyXlD=!-$FMe-'
    '^?l1;HRF4A79yT$Pj%9iL9|26L}@<9-`qbf7iI?FAf&DcMr!Kygn98ZQgP5UM8HxYqJYU4KG#W%0B45dSUfN!CQLvY6eK)+bDpN}18PJ}G?#I|<+~k>jAo*S'
    '@}qQ8eHiJ0v}IhpX&AgP_J<#`xvB0w7QYR17AkpMkK6sVNVDo*^)6izt-_ONfQ5L{n4%Z<E)GXFV2e`(Zk-%MXs^Y2BkmQUYZj;^s%c494wojdl30BnQGH5b'
    '#ft$TXwu`l4rH~8iD8Q}OFO}dLN)wA)exl{>jHItOpnSz?1&mNvmR53kSduuvgX-!F$osZ?}9do!NXx23X$>QFjVNg2DSwX-3ase45>R%Piz9YOuYVCe3szN'
    'C^Gq4NT_e_41>pD-h#+NGNqK&1E%Z6fr=cPB5i8GB*mURj|L|$#$~s_4`FzCqL_(T#3Tloa3!!a<Lg7Dz!a+kTCk_MU@W!9>ejRZfv=T$D(Ac##frf&$!7pt'
    'agHIUC3-5ah#NXi+!Ae&wrQTuG5I#W2}=1^3pMfDg2jXvTW8b51yG2`ySH9-6U{qLG^q3FBH=|IhEKV|^#$SpRl^`Ihl`4Xh(g!L#s>KSY|uwRL!bB12p{>*'
    '+jxqnNl#U*$^3R_bW+^hs?1RZP<72LYb3|}O`pXF0cU1er7E)N*H5|~GbbJOaY7$>O%ea56xj5&;kD}C=bpW2wW_>~9xw4sQE43VOp_Gzh(`>WHRVMB;aMg#'
    'mJ51s;0h5&H>y~<2x`mdHRJZZh_yS#WY-A-qt<eIZWMIof9!e=uHC5PO7$pa;b6(l$vE;=yQhM;2<?WgRtx_~Dv)FMzVzDi1^yYW&#s9&g$4-><<xD0U{oSh'
    'Q^gnuF_RUoF^-PDaD`$<i1<JHfB&t2_)kB2gnz%Qg|94T{vbI`0Q0X|{5m?1nCms~eZ;!}&|S@9#8bNTN9uky@>x7cMo?HJQ#T~9RJpZe0HSnKTXK;zOKB{O'
    '6tB6&-(6J6)V+-#8>+H2Z^qM$Bts8?crMJ&fq(GLPhRdHg+G1s?hlUM?jIZ?mOOv)6^cLI?bg%X?h|>lE{%$AJP7nHnp4<%g%{qByrA>=@m8m9UbH!hrVC7|'
    'Wmm_$s;X|8EQXVcn=w^0A&T<F`#cM`VxGdec0P%+_p2|^v=_7Iq4v!vjgC-Okx<|B8HfN}y&in|L%<(l;#?5%l0eo|XEDg75=?as$<hiKm7NP=y|)ql@BXu2'
    '%)a@xN00FDU(|?xbQtbqIs{5mX(vE&T~sc^fCw?C#ivH{4lMAzm~!EVPquKt0>@XFq-c$x@H5Pqq~jy$YFA2_O*F9A-Yt1z7#*;>%8YAsyD&_~GZamy%_BIS'
    '5VF89TPL>!M}eVNJa14Kru7p=tHQX;?N9=f^_q9ISLo5^%<3gGYiR|`V38RUE|5)m-h|I^#wi}WD`5P^+IC=V#n-KRU4^f@UzdR|#v1}~rAYvmA}~5Plj%mJ'
    '8hEY}>=?D{&BavIf=-PeTbRIgh8f1A+!f)4ISa61oeJdRAm!0v85eCFG|(1iY3`Ecz17zVuM5+Kg0R`N10q<oK#2a^|MIW+f9tm%J;J}gqzTb+GI;+wo@*ME'
    '#frS(WgoR7u_+NBghY*%m`%>;Bt;@C<gQ_)lV~_%J{neZmT1o;ld{#vlDfSM-#&XQoRQrz#Zob57A1yM&e|&V(Ib#~W8EQS_wA*TYNZ_O)LRkvv?Q|DN@1|#'
    'zm+Wu@8!xUsY<_YjwrUK{OnT9xGE&b9=hd<U9WI%dwbdW;lQ|N4Az~Rxm%|)z_f%c?>-W;tcsso5^Y@=BR?dG6~7c*2&ZN~BjcA{%p47C7%K`M9l~ok3+#0P'
    'y1o_@B#ak5@2VW&4EqwOZfW(@`)8L+UjsfC)4_-VyF#EP%z9p%#JMu8dS$?rBc(B)o1B&H*;0v#1`s5M1@c*ay$*B`HWO_@cvsNt1LfYY^30;DMN(}w(7RCX'
    '<rh{^UEDj%5M39l%oeed`sd3Jl~MqZcu#RCL8r6Lww#K@5tNmajCE+M<4i?&OAukVAd*Gc4@;Yt%tk#q%0hw&l>1K6Mr!0TFc@^hS>;ob+r4r>W-b7GD6Xst'
    'qCa$K6l!pBXWc2<ELZ1KFczb3jShiE<h8cCLxm}veBK>(-mA<b)N`Ptq^m_Fv<Jyrn3@B#XoZTc+WsM=&q>g*PTy2byhRUngBj<f6~HL~UuZJji)?Arp{oUs'
    '1bd<{?vCt1IMmC9!$jyqq1is7jOc<c&f}!V?UlzaHDE0ywsLc;+zMo)nB%`;bD--jtTo>v`%+()HU;;hS$R#=rzsgTkT4Qcpq~Xgz`0>9JVBSpR%hGIUOD^K'
    'jfgF<nkgw{4XhYS!x0!*Ut>UL<Mb5TLiyLE1FNzh)+_c>)$okHI8gFVA!m+yjxwUou~!L~ihGxexp>SJarLz_$9?KaP&o{X+C**x^$r$BWm>V*13C7xox|>Z'
    'luS^w4hWBE8;XWR3OmO(8A-DY`L0wtF1mXo9wVC0RX3E}z-w8Ab@gIa!GDc}ITE%`u2A>j&TgX;uhP9>&?BieWujyp0pHaoSugj6o}cBDXgvN*Uc6~K(Ugx3'
    '7}{1xsgz|Cz0c@Evd_9j)e!kjoU}Vn`i+y_ZTN@3c6N5*-<@_x1eDlq@px9W)le-ppq+;KxUszXF*={#z4=D5apyzbBSEuUD{yk?yn${xQrGNaF4g5&2JeW2'
    'Yl8>_Od$W?cjWao`cVAhvR*nSqqSr)%j4)2#^H7wG3s>6V;6YrIk0op9di|odv7gX-VKi48Z5qzFTIt0i23)b`G1?s7oSB#!f*V?Kl+pZ>GvKz!oT0r`a5w~'
    'U^r%5nE>O)OYfrHrC?-Cejg7#U{p9aAK#cYNv_DGHk+T5A09gbp_d!S+YDtC2i+w+OY;PMi$7BW+{RB3JJ9-u22HY>&vU*s3V(p;jSA0Z?UlP9Ac}aECAl#J'
    'P$E2KpeP;>UJzw}m|S$*Pg{G<*3+G*JN0JmIUckg-9JYcI%f$w33~E&JihUmZ-d6BhUp;R)aCL`_K~A04WMVf0y&o&#Wzqdi-*sW!MwJHYo_52GF4j1k_pD8'
    '5H&QQMdhfi{jAlFj`zPh3=h8EfBpLKB{^rdf!KIJX}tC}?&>E`8Xl0Ex+wYV@a^%}T+Y~JZG3(B%50~zUHkfQ|C#w{Z}&;fd4D8Dwus9p#~T-IzxFXrOKaGc'
    'U=`nF$r)jTz%k)ui-<tM?1rPF2uDKECPI|f*Jb2n3;y9O1Jy$HpE;|PZfCtD*VQ(tY7^8t`q1T8st>YG@!tM87OIbIqWT!7sr5Iu%QPitOeeuJc@VgE_Wo=('
    'zV?R6h&<%xUXF=0U_;9@W1gb3*w5+WFEL!93tzgMi_&&OHn<+})YZT|2-Hc+&GI9P11h)X_OVuT*A0-b^)SIx49>E63e%6^xHyx&`9<|P`h!&^{Wr%_kcTsn'
    '#?FZaQ(%9IK1U}-njq4*Iy!53&WGh3td*BrTS;?No*b@(<L^|}6!s2h|7IfMqLN884g7!riz?AC0%*#hF_nU4GkiH;s{788HTKKkd@+5WQyO^o@pDdYm`qG7'
    '4=W~!k>(W*s2TRx<0lPn8Rz_cG7GHL7w__V-;yy!tLPODsE$Y^p48doK&mt06BJ2jyYk>n3SeH^pi;XU2fX4+fhpw_P7Z1yVp-%ym#i!Te&fdmM+raZNS8l='
    'E%@Ba&w<?<EapoI!^$-74evY|0$)+W_cck!r{2rtee4md46HL<u4fEZ_+L}3Uj#Jb5}o6J&b%pdv}1xmvv7OIa+7d*$ak~euCUGy(K=h8<Rust$!u`~Wn*lq'
    '33pOs-X!rdtDY_<Fu+PuB&ggA;uYuD?9N)WuN6#dPWz&ibeAoJ^J0LaRdmhM)waXQxjEHEGqO#VpARo9igz3gi-0s*F%6?$!HD<LwQY~^95bZNFAUCHKx30O'
    'FU%P@tSJ+(6ghZ`X~vUe`FU<?2zEI7tE948<4o7BpkM5&!`mz_Bcw<w23fgyN|%*9N)yCL<Iu}6-2@_=%z*&VO`WmGDAx0wvYA6SWQl&|UM-D{?pnn<8DemO'
    'GcYY^g=4bBt}iowZ8bbg9MxAP6vo;yTkTir;H|dG;pT=-3nSheUV&y18p<KmZR1mkYT|Kf%j7s0EqTj!@?zwSP`cRQEi)@rj0dW(n5Gdr11o?)qk^#6VhW}v'
    'X_&iO&Apvc%}>2|=>mwH2qDJ<25%9ZEXMOhBtv-O<*CpQDaMgF6TzZ3%Yi{ezA;)Sx(eF6c(Tq-S_sk)kN9(MyN;~h6L|8ZUXjqBFnOy{FV5^suOko{&}gSz'
    'n*n_?Vw;@_&(FQ~EnXb&;yh{<s<`EJoUX+-dN7QP?gN)W<Tw-Em|c13@MeWcvR%)FkD8^eY+$eyxBH1{pzAN^C@ghR-I-qCQ)UX(3i1+pB2EN@Tj^LRob#N>'
    'saYBt&8qE~OErc1)T8C}+67vKxX=bV<XCyj?V>=IEK4LQ-t@&jg-P^~mWz4!5N{hmyZUuU`U`7%oqJ8MbI<9~6s$?1BC(42mgk7d#;Q5G=eKpQ`EA{EemEO@'
    '&9?RY9P`<>=L5LfD4t_-NZXDfb=Cp-_OdZK@QBu^JgC*Ew0$p7@7x!aO7kLBbTzCi=H(pJ?mdA`L#P~-Yt!N}<p#DJ$UFA}^4>jxT%OO4L4-PYnolM2EIk3J'
    'KL_PEslIH;bgd9jD!Wy*>y?gSHdM_R!IeG})(9Q<Y1=`bWoL#vxiKtrVE9G3F35f;t%BvTQ2*f^6HF=4a6c&eD##!#_AgMD$Y31_O?06}Nz*HVi_^bFGw84X'
    ';a~XA|AX42NBH-94l`&-hW+rOQ?!S^#OK~cr|Y%1O=~D(2OAbI&oNI33`|7!B7+PC8_W!E2z4T=2#yYg^h#MllAN=J<uSr|j`=*a97l<my;K^l^ms_sFwae+'
    '6ea?xEvA&ABpx=spC-fkd5(c#gL66l_7@(7Z+k+BP8X+`m_oJt88GX_Yw`39!)F+d79?4Q(i{o2Bb{>;YHdB^s5@272bns#vRXe?zeFL)C~Dg>w>mC6>23?e'
    ';0_Q_^xN==knQb0aT-eD)eij+)Tw9{MTfU_%%X&F8y6kFq#vOr`?lj!lRQrs**vy1B_c}zU`vh}p%^EHxG_T;VMgJ9YNgj!lC)bbV-v3L6#Sqi`7^8~c~Fk*'
    'Jm@hV^cW9Z>;o6;=3*5i8LV}d7Wi3p^smX*KK`r!$A9rp{^FxY_~*fse2^uxdA=E)on<jOXq>@*4f7aJKVWS?{bX~IXPc+VbTgh_c(d#Ic{<%fW^Z5C$i1(i'
    'ig!d+(1^1bBa2BXKA5}MXmf%P`J-j{*0Bv{9mY{U%g$hNauq$CU#k=wKhIO)HHWz%$H}Q|=Pjl<V4m`NMoxtM9FKv9hb{^Y?~TZ~CmTaA8_Z9bd$VD!2EB0d'
    'O+fQFn+GkTYymbLgq%Y;tT$ou=@=9LU@jY++)0}x{o3Ilz6FNi@L7mo-o1Er`1%-GKR(xeTB&v`%+H45IBT_AGPPmx#ZK`>G{iJ5S*z7*SHEg)72Z|%RNd9G'
    'zDk|-1*&VT5@-=*i)kJXldMZ?B9qmHgZ!eaR^S9*^+letD!a0%I2l&e!`*{uMj1{GeCsW3bu>x_KvuinDbQh=UZjE9Skp@+3iHxS%KtGNHu1`!=jaj0`;@Rv'
    '?5^eobUwNc30xt7K;oZx4o}41M%Pj!9iSXPJaJ+WkVDrLkuJWbe4%*4x=Pf{5Da3vL?KPp5Zj?&xnSNVl`Xm}QPx#>=cC`mIHoisxDEX}=6%6^;W#X8XqJ`&'
    'Z!Aiti+H&ZF!Em|Kx$z#W2*3}=sz|{OV_)>A>SGj!C-U-Vs;EX31(e7MUzy|SOuC*Lt#NrelhLVv_vX}pnciRq3T;_6%<!V!2{_0#1}b({C;yWBZ;VRUk)Pk'
    '1mu-WR!G$YUB&ZXzLW1MHxr(r$TIu=LUTW*gVcK#529;XgAdnmNY9EDeu6LoFnjxhffxL!4wXjZG@1tptP}qNIG1@kYx{j1g(`eF^C=mQC>77r!8sPO+djtP'
    '$LVaNEm{^UapH564N>GJoE+xcG5#(N^c9{k-t)FuaDvam5jXXgj&KW1#Ux$DHaS;n@Ks&bI-{&pGUuRKUL;%`jsSto-7iuS<|-4CLE#D>R`^UD6wS$d*kge>'
    'HN<^I<mDcw?}fXU%KJHL&Qq)#=w?J(decKqcRd-FdY%mL(eZgK?hx|5KlVf(xgp-dJzh+vCdZsiISj-96p<qta<W<AhnSeL#zCA2n3BqM1Lnv-5IcJ4-JrLa'
    'gzN1l{gec&erqukz=mKi-uX2=Gm@2^UZW9E422RGH(kB@I2@Q92nS-epW|%xQ)BA>#>R$EL^PyW01XLIsb7G|=LnKgGm~f*04ssH#xLu#Dm9CqTHJ5~bD%o{'
    '^1xW<29X$cpXxhS_%eff;T&`5`ahe>#D6#x>gH#tKZt>Fl7FPXn2$D8RxoBVha+*0Pa>#E2OaG3u1wRL6EaACtht|^EpQRu(i;$)`ShdkNyRL>c*Um9{0kK+'
    'm@*P=A&~I|+5jZ=If|^ujDtvZM!y0d%lJT%&g1c{=i@W<Jiw<qVZ9fKI$e5EU*0Q&aRc_&vVKbeAO`mw=8MTB%C4K(h8o)&&NzB-^evsRYYk#D{NOP_6U8CC'
    'in8~Z|9*{e(1E=n4s4&D_i$NrfGc1aPzg~VD&fu-&Cmxnqy(ulLx^vilF4XL15J~znGYsI<7{S9Ww8=HDr`Nn@VN!(oS+Jmllr0-!Vg14?lAPbBE2MHdO>LV'
    '{r~#k{3rjo_2?1){jMQ2soMjHv13EPK3{i;!ASBR#7wyR&0|=Z`88|e;i<9+l;tCJO~D5WN9R$Fa!@%NVSbJoX4S8J>=38qrJdsEA{#?f%^10_!ZY;0fP?#k'
    'Wv_;JOgEv_p6ws+9~~ZtuMUs*@q3qwpAenY-{7A<^7}WIGy-S0k?YgEq>u^#d^cE(#T!MMXW#o~eoELwuw6pqip9LUxjBd~V%WxT?q<N7rgX>R`z-l>b9IX&'
    'Ku1&p6q~%fyezi(H`Z)YT-P|*c{17p5$?%oul;27w6oV~N29INr#tc9_SW{($hY!=%DWzrkeZ;N92ap8y@~R-1=(&jI6neBKL$WoLo_~p+S++C*owBEbVj4W'
    'Uc0p$Z?{^V!FDHlGJLYL7DQz?8l1Pa+Na(dLAUh15*u_~01*SA(jbnI{f!KQt_w?uIJ8@v&_lC!_=m^u_DLK(`1<hR501Whb+lq1=!7O=n}+-J=^z>n+B==j'
    '_I5Pd+u0iJZS4%>z1=5+;ogpUzo}WbgLQ}7r>D{O*3%foeZaTTuoXQ$9gX6rdxO*HG#*sc-NL#%+wtk31(mj<)Ap#f_2enE74PoubfDwCopwdtZLGVywLNS<'
    '*>3M_?R192QS_v>v%7a1$HQ${s#ZIys7q_R{REb=y*=D+k9M{P(9CJ46YrdE!;tp|TZ2=xE?u%kb*W}+Yo|Tjc>;Yrh1s?NINMv%X!N8VwL4GtVzZ{$O-<lB'
    'Bbx1K^mMQT=o80VyY21K7StaBfFnSxC-C2rJ;%3*(N(m&t@hL5Uc5CN?Tk9xot?eGa5#Lr_4E|Jj-Cvij4aLpE00?7X!o=QAnt5+c1K(B-e?4%JKf!D??r>r'
    'z*$)ifA8tu)@geOMg+&$)4|?wyEQu9?riUFJsAyQ20mjq@rcHTV3ZAs;ilZ|n6($^HLVF=&cfOsVkl9SA~Ys+bt<5-D&}o^#e?7ko9mwxwx1Q^jcN(|ZX3L1'
    '1th;o9<WT#J=IGCFP9Yq=L*KC@z}yMs4tjdQAcZT@NCyt*(f~@u=9?s%aban0T(J50hmqIU{MLfib=P+MGj4BEJpr~Y~~guXI7yY8j^BcJ-~)z0#L=W0M-qt'
    'B`HcuDcwcMPkIh@(t6{0^=wB2>qdUNZFpxe%^Pn2))rD^gHq)<pGU9-fhPlw3gtb#T#zp5F)&ZvpR3kbuPP}Kwf!9AE!8GrrWuPaWedFcYeI?d@wU+{U7VeJ'
    'e*nBLT`xL)&upAbF{KC7<jb#m)@VvMrWzGgx(8f!v!R68=b&tqQ0R!x*a|!GY+<#czB`n%pe-`g12oKuHPm%rm~8qo8oxK;*$@%-G|i&TU#P>e9}-nW^M;<>'
    '#AH<(dJ+S)(Z?9nNw=?OQ>ix;t<asLvURIV1MVbg>kbMRGa&dR##&Q>U1>D3tZ{<khcqizlqv~iH+LixFK9%fI7Q)+?*pmkWNM|y^g;Lzc;6M*&gADrdWF2u'
    '#0I%nau!i-^F>{LcFI?Cjr4rjsfnWkyFjz(a>+oCp<cWw0efsNW<$>Ph4F>vEXDit_-dG(!BG}SYCV8<Va6b_pe>e^8#EUZ98TbI^XTa10l#U`>7qCD0oC=O'
    'fH#mOke$S>j4UPU3>H&T2PcS!BX68iJP}?kNwTgtj?j<bn3+tH>9R>YOt4ujr8c^Cv&vvjVNftOF?xADxSBz6+d_?o&Q%^?4dU6{JERX_3SBLSqG>(;6|HmP'
    'bOx#tWND#s$wSFz<E?mkS08wabegE9ObmcGhCK$$3v&#u5f<jL8*SWC5eaAHEL0amn(ffA^9AtPSApN0PiDSQI*cP|gv3^3Yzmligf&G4VaB*&Y2r2euDFqZ'
    '^0@C|s^H0t5|Bv9k-Z1FHmGFU8|ad2HZQ;;(KT5-vtkfg31`;F9mG?k53>2JjS<E@E>ofawnX1#V>Q1G3h%x?K7M=jHQjiU1)}L5jMtH04yu&fe#f8KY(uD9'
    '^WKgBO&(_(`)6!K@n5CiC*yInxzlU`?AYp^ZydbXc&5}Vb^mt7O^b?<@kE%4!7u=slS!N|=Dpn(5i~d@X6Y1Fs|KD4qIW5X9{QT#KB-Fig?HYlRF(5UHrW(2'
    'q}x+`0@rlD0i4g-{M^SZV0CAEN{fW2XshN<5hRJMP2<r$y#4AYA{@KstXm-4YOu?xcQJ!6U13P-W|Ne!n9k=tiUMCzf$ow}ApNn|>Mjj5o(+&J7`<$o)e$bO'
    'W@~8+lhcs2qJ>rh^YC2N0DI2%%&za!f}}M+%kUQvPKoP4ae&5%5q6vH(X9vTyGf6zFZC?A0Z7oVAJs|CKkT7xV>Djm=LNG)@jv&4Y(FT7?l!@S@z<(bt|RJC'
    'W8_L>FB)Nv9&Mklxx9+36RcK|P8{zjf%gSb$e<MNSu4p1i#LH;MjNgvC7I8wA@8z)gSZ4D96N5~SwrVf*C9e-7q%EhPkaX$P0E+WlyD+2&e?jcYBf(j?|VlJ'
    'x=a|aa=kXO1J&E|7=jfx&+Vd(zQC*)MeVT)t|&0|(#>z)y(Hi?Ji!WzU&l=6#jiI<;){kRk#=aK41di&z{10)3w=zOiQ9TBR?;lFn4;JeR6`WMpM25>ZUG4F'
    'kdH9rUirB1-RNT<7X1ds=7OUVi)s~jrH%ak@mO;*5py@d-E*%Jukzcb_Yz>?30CSx1BUR(RHonz!?W)L@}a(n3(O)HQC>g=-q7T<j7x`a%(fV5kYB$sn!9Z_'
    'r7f|1yRcauFBk5feebwmco<`1Ihl9GXxkyPPGY<hJZCl(k9F~&c<&oMrEAXkeZ?i`f!X(_C`=P|3CxBfKwwjnhgm$1(2o=j1GBX{y2y~Xj%YN=Sykkz;lU$P'
    'AXMJ2XVIN+rg=0H@h2CtXaWhh9<zywu8W$D;WZ>+dg7B*4nVYfq{Wq{NrTE9N|jyLg%-Ac$XG*(qf@-zR(N9xW#tkzC8{{GL0oVNt3KZ>ZIjbCjS${8V|A5g'
    '2h|1QO5KtbthRJb-cRvR787FMr%zx{cn9Zcf)}vEj>Tk5#a5f|<7>QBR~-hZ{e_XbRj<O#%P=PkhOMy0%I0<0LV%SjR-w`?*C;A<YaHHqKvy1<&lq3JjsEDE'
    'w2KZDR4|(=F<L0|f_tYn+F0SbL%{oA|NXz+XXr_PtK0IW;M9p43%Z)wb2QOkUO=dPzm#tn-UPis7bRbG5Q--0nrk%66G{el+x1%s9id|!L{nBjD(krLv0=^K'
    'VoK`c_f&Y3E<fZj`O9<o=LPC8rhqU>j@6qEJ*C^v+AJUxe8waDehEWW={_E__~+8ov7pss?Ke`{6{_8a8<WaW9zvgvIyr8;>pIf;okO$g(q`+kVui_}f<Kg{'
    'mvDaqK&IZD+QgIpvxfH>yZ6Y3>Ml-Tse`3vt4v(Gi+@-N&VmRmwR%*~m|YZ_gLHP?2}(AQMv<uKSCvq<Eko-PhBeN^eQ6+Ihf}fdM0}+;Tx7_=;6!X&OBECS'
    't<*q9&US^WNi2UwdutILxuFzLR?!_xAmT&d4m(qy3gu0ynvhHdAdT$7xj%~)stjOD%~-`oRtRt?wDDBXW1xSh-CFDl_N|<%ZS|rKK7$TE>$?qKciz(GaeWWJ'
    '&&o|%oKnN;sS=jf0+<!n2k+`FC}Df2O;{nl;NVQ!@GC-JgnXvXd_*Ibu?NIge+Z?K;1jf_guFil)e&voh31f%@aGk#<#L>`f*Z3#bnj0QY9_?GKb!i(HI|Q4'
    '*_lk&fnqSSD2%11cT-;xInzI>uNc{9O1zfzP1x=Q4@q}WW!YjjuPFuunOPyZSzB09I<9rs-BhwOtTO-D``R}5^o%YHF9O!hm}&;0UrDo4ILJy)(_Uapcj0C}'
    '{2NwF{A~I<Rca@_<O0VT9mq1yEb@&p5|EpGXXE}Mg%L>JTd~jT4D%xSPK16CaW-;2sB!pJqvs<)9+z+qD6BzU48OJ5Tr46O)^ekD4n|_f%@RB%f(n^h_Efto'
    'J-j#lbI>~_-`|<w)z+m#S~nSDI_<SNAj2U(%FfKFHDw~T#Qb|Y%xiKX6{pt*5krSS;Xb^1@Qlikw-{j93TJw(y@3m`)T5lVF4A56nR1|?r<CAc4?%m><D>fO'
    'u5mHYGl3Kmy#T~sbSGCHiw(mCuC7NViizQfO@&#y1(25aOLWxjv@Y9xPNX)Al@64|W0K2Q9ADZ2Q;jlID^Gatbl>zU!Wjt}2bNV~xZL*hR3gD?JVHN&MJ}lc'
    '&>p-Ne3b(}BCCvMR?3$iyVt7+1wEsFz6Q5w*!4O1$F-=G`&spglSUtWTwb<YpP29Zmfo)iC3y#E7kh03hIV=dMd358VFV6F&E>XG?3neh5~r1MB8SsM5)|BO'
    'LQk7Wi-aMSI2DRq${=|{kH|p^^tE9gef-?gpcvx>mV^~ksHzu3DbNF09DUXyNnn)~ja?9jPyD;aLtxx9Ec86!z=&dG#Q*$%{rdlS^*fIq;ot8YkrCNq8nV7A'
    'l<~-Rz9cxry|?EPCV<%1K@ci4O?!KTFHtV%RideJ)g27SOlI+iMoCcIxH~XnaXQOVv}ifvBfd{&6dzF;^<dxeQhWF2&9T&!Esq(m3OC0VkQJpKIU`k*vCg=x'
    'MotnoNiqm=ZtTJr=I2poXV<D+&O#uoyR+O_QO(Yr0jm3Aq)+Xx5z)2U;oCO{ltQV~YSnbyGe#gvtSV0+Y;SW44;9M3w-XM4WGv<(v=h(#+QFOG&tH56&CrTa'
    '@bYw&oaLL8<74w~ZF9JD(UA(1Q~{y}C0jzt7ggtKdYi?gIMaJrZ400|irWNUpHjI`wZRTa>M0&m%KJ>Yj?(UpVfM(fY8=8jmxGruqSrj8H3he$WNj`2+th3w'
    'HDMo!SimG}GC6=)>KSh3_BMrIBiYbq!EP}I9wFVtG3Nm<A-Sn=gkvd?v2N9to`!_?Jzp=W-|(D?o_fv=;6Br%#To8AaaH6XOK0I|oXiB)`R+<`-b9<l-s~Cs'
    'R4o)2aFNl93v67W&b{q$duJ=$-D#l{g%Q)-+k5gP+-<eP))SFoh&O`~CLh`{@?w!np?FFv<OX^wLf1seLgNAzW*})|VjfM!suD#Mu|o{TPE?>tg(;cHfg?=O'
    'blsfI-B}780ttwb>a@~{^pGtRLkG4&G%xuJm5`uk^}HlcLd^)@MT_&AD!P0g;|0pTL@0$R4zh*Xjlh?o*HRhBc@vQT@-fO9*8194LY*w2;ZwbFwF00z^M|>V'
    '=rlyb|J?IMnCue2p^ADG+Pb8gGYLqbaimc%ptA6XCy}$~-GDr<3*T<u(79|{+U4Y%Q66Fzx~rgm3*sYrtdxu*WUIj3VqB@2VLGV5f9#VVrya?`Mn-(ZZmp=%'
    'Vb9iE5v)2*Mx4XqhL_*RtdkAT5PVHGoQNMjGs#~Ga-6*Zg=9W|9VN6FEwRIeOS5Q^vp^l2c`*#gms`fD-4UuBm<VDZ)cg3lYcen#dm?V$xo#+k=D<J1L-K>q'
    'Wh_no_O@m@SybQ<uyVfkFHW-vq+RJ<TyQLgqQbdY`I!htu=&@M*>hYt)do!clG5&}J7s#R?2EVIv%}{v_m2;sksFE-^}03tYN1sV?7D0+!WWZWU7!tPUz!E*'
    'WM;exB`PsdYVc7zTuf$pAYXBgiRrx8sawX~GJz<MWa*X%d@>{D@7@dtxBF(SyS<hLKQ<vMYX-TZ;R00Z4YlK}Gfhf*5)k50!yUflKr3;E#S;!Q7`e!rN(xov'
    'B&BTOTc-XB;>epkgy8|>PN>OIB7e-8gO(&;LV`ZA`nBtLLtvo>>`X(f`eGP$TYg=+xbXWX5Rq)2;|2<-jL-CrJev3#D0qm(FuI76G4eCzL)y{>`y%&;(!YN7'
    '_8Tu64B%-7i=9c`iIj%*8CM2|sfcEWl5!C%QEA!uz%@3Y#)hm>k*ic6jfS$oKuJs#ISUvkf)$0u-qeCCCRYfm<&uUAp8(Sbm%|=%pbh4C%m@rgOILDS1&2tX'
    'lu+2S%&kP?vQfNVE+W1t8-fa`GlWHshv0&xfm#-b3fC0LKB_jBbB{~7$ajLdlA1oCywJ2Kn(IpYL(#Zq*j(pX9E*X-Kpe$w;<}>ZZJE$_7g+)XIp)@d496mu'
    '77{B2Ku0qojS1&)!Qx`%+!Rp*Uy==aHcdB_ZkM%ybg^nkTS~=2z`}@@RxXOcv&(BYv2jOrr17&5<GzG6H(S6nPMMX5m^i3-4WK<V-IGd#rAHwai4p;AY%p$H'
    'Ah9MQFWpwLgboCCVA%_u3r*k|(pXpnPcrw$AeFlF_D<B$ob#80PwZCBGoPok4f4X@V2JRHC&@Gr9H!OY=Z*o?1u!hrWr4<tUYC(BVV3*}a6FD+x;x+#TLUVE'
    'R*N9pYh?;~zza4^Iq^;{O8xcCXUd2#M6J)ty!dDD%1fVrhOT3JD7TLI5W(TN#%ra9x`&A?42S6;3_m#alVo<#2m*)tCZ~e>a3ff;T{eoXldkD*176Xlpw*Q|'
    'ri#V|>$kM&GJ_&p2JBeAE=0LXgINxAP3MJGPcnw9GDk^0kFhNtKfZtxlst;^tKKK8)5f3thyVER|IVXF`1gAiwu{All3Zzv1(H9Vr0E?{n|)UO0ns{G%wK_s'
    '4wQ{T`pADxGNJ;!zp{3UUnR@LFU-2|?f$zL`<#dO#*bQ<C8pKf0sbSR+&S%L3%_^pd#BmDUDImNGR+Q`!&a-c6K0Xbq`TM(v|&Iw1K94}goFvkp$jw<6oo?7'
    '@s3MUPN0(nCmH%k=7Bre2l^^bNMZ$T+W@w65+OSxb8)EL1Q^}SFJ)MST)4HuA;h*+?ca!Rg(#7czs~xFx^}hZ+MG(mTT_qCf=d-x>eae#l@I2J(Q^NaDz`_h'
    'rg|ExRVpsX52M=tFk_R;;AGo|GE6GYPs9vhhnNpBk9#`Hd!5D-|4?0g5}i$x`C=IN_JA{EkVIpERZnh`69J%pb%UZFh7Lrh$vBx`_mB{-Dt{Nlm2r~R*PGd$'
    'S~n_E7xkj8;{JB4KNZFd@@`vg#vot1UwH$@Q>Xoc3>X5J;j#GUuppSb01B?L4?fJCL9l-?WIPx${=676mh(6FHf#LS3=)=)*1EjdgBjyj%Z%YZH1B4_D6-PO'
    'Jo5#8+J=i_-p4<WCX1z9<T7&w*V|y8fM1vR0%waW9Nj7n8?L4v>>PjE?Hrv}>pn(~4&LSUXT-uG`EKurH+8Jh^q&Wthm>)2n2aP<4NRlANV0fnjP}8D@?hR@'
    'nm4}w&;I59U;NESkMQrW80L+lZk(+f;+`I$dJ5TxI4UnXi|}Sgx<Y!Kp5+bmjA0Uq#W7N9nDC9xcQtc>u;j9EOw`2@PMgI#mIg!EZb(&-63i*ZY;HTF>pQ?6'
    '8`>0c+fGqYL@|Dn3TXE$og_`GL$wEWp?j+|wrXo0jqBC$rPQjf)I<vR>WaoFzr853c9WzaP)+dWVmX)dYZa?CWCFgP&v~}aDO0FEVa}7Bq6&K+r6t14urMdm'
    'J4%xoaOt`_tT9!js6mm<43HD)tlG!0xf9B_sjna7Ta-}aAZqrpCiTWw`?ee9c0Tvo^<|8Q$#+2W!^Lcj;is{Xtz=hX--|2e@S<Pwg7g@@aWpv{MtH?j*9-W9'
    'HTYh*Uu~oCg6~1A*LBm19V_3xCIKhcnbVG{y~{Oyf~C<?YGz72DVOit<(I*Uu5vSo#bw0ImAVJax|7RaIbfkkqY)RiXo!+DDrk^cn!5}PccM2;Mk7o!L}bNk'
    'N?QvUQ`|Snc|Z1c!&a+>e+)$~UAREb`*F`}%X0X)Y@ml%5sYOqM1Bn!yLcE6E8rv*rq<q2cMWo$H={Uvf?I|8QGeNZUQrWXg32?z$jE}35Wd{QBZ@=9levoK'
    'I72u;p<Osa`zL&P1$>Ia{gtaJqe+2VzKExI4e|296-c>5S9yV5m^WjJqUBo}EcBr1qM{P#M{TSnW^-@}H?hb#1)!J!D~_euG97ptZpNj*Li)0aG?rT+;{)sh'
    'EXI{dt~GIXTmt7J*W3h3>qKTRn7Uq&DImS<n>W@WDbFl2-Z#lOw?q(v>0sW3>2*L?{H2$Lrxze~wXBZnW=Eo><<y#JjHh5BDk<}?$e%%ah?7>|dyL(yTmAT)'
    'm=5YL_!3r|jL(O_*Jx**BN0cKG$bHmKn2te#(~j{0T-#+)o*2XYCK!`Egd=PnJ#JO^}w6HWeH~d_%d}oY+>ONm1wf%eJRZju!I0nWEJV5F$8>3%mY`sfJQy&'
    '5(OjdY#f~e1L)fqwDN+UaQFQaU9*qUdYhBNr?_o!(TziPidH@}iUFq^j+6KCI5|(#p<*Q%*anCdUL&&N#3yY38sR14ae$prk{0};P?>%}J3NH7u$P1wX0#FM'
    'sH&JWKgOAW(89M#E+!4RH6&}9VGajX249^defXd8$ij3*BxTF6Xmq3*dj(U=Fg?&Gi^PbWnFLweEFi|g1;meK(|@bi9Py{PhdIACUh$YmXIUJV&+6)v#WB>S'
    'M={cJae3$@nIjAcBD0uSKUx&orr4lu0qzcv6bs~=#0~=QX)Olc$%zj#xhk1K;p(n+Lap$iW}RB*MMFWr*XlMf={bx+BEhB{C~WsYUzyv05u1#!&>;(i9stJ0'
    'fm}%FtFJply##Wt<6b-7Xm=VIjjVsJ+Tg6n<DmgO$uwvkdG<iW&MMpuT({7SpW%L6XKxiZ)Ut`otBCpS2-i#OVhz@{!BHO4J2`(fCWYHMpFIX4u!pQ}F1L&X'
    'Vh?_T_W~3-Q;U1RA_0b)j~v`%PY2wKJRN{hD+%`-I~OTUNhq*#XrO_^kpsltt**bLLBTK<D)CBTS1VcV6G{zWP71}OQ5K1V3k$TG?NuXX?n{jg6Hoxx_<cOZ'
    ')>)6#@Sft;d&pqU(?L462Zq#N$ezM)MUodf7PogK>_s6QrCo$e6y??RVNBg_8dn?(Bq1o~f|vA{g<|t{=mjOx#eeso{GWfd^XL)&{qyj|FvhUn=olZcWsxal'
    'Y^3afVN~32%9+f+;?za^!)WrMuCe#hAvta(-Jf1JmS0V3*^A8|&FK{Ry6IH3OV7{@#tCiIy{GW~CFQJf>9EX_ufKT}9=zN?IyyXp#i5H>xDcnjffbvk*#wsS'
    '`&cF0;yhP*jJ5{Uea~@6S^dxlI(t#_X3>@S8PZF4hLEL+LFD*UX{!x^5wZm?DjwEq$M5!Eygq!V+EVE~HVE5*;9Ycl(-i^tBPV{N_PYS`=BsqL7z_6R22a<3'
    'G(R7p>qSP61%(>YL4emx80gr10n0xD0Gn3_aVdJ!d?VlYm4`whyEn0y`VOi?GIPe{N-iz<NEu?yV9*4Oo1{dco*s;8r_z9EQqe$GkhdSfF@e`Wqd8GreWKYx'
    '{w=R7Y1NF}`7}vHsk1BsWFn2nP&aOi%*iA=!;_EMab|955o{@$N}=RIUz(Jqo{gbX5K<{e0g6fG^rNEPp7^q&W>GYkgYG)6JrWRY!e2^L4RFr$G=z3zS9>5D'
    'B$VNbU>@*PN_qLYR1*Z137t|>Fk-M80s<DR<DJ#j2KncRK`{KcsI4k6FVyUGgPRk*7HsjvM)WlHi%#WYmBHm4XP)0_e6P(xLH{TM<Nb)4PQ_~YudKgd*jN)2'
    'Q=&S}d5@J`(0m@xf<n`LAFQ}3NuUx+&f}mAGTJkEM-ldjN}r$(OtfyHaP=VXC@3$$+k_Je*?d5NUb~JglmK@VnZ~+ku~xb|3Y#!GshM{mVJk~n+5<W&^-&xF'
    'F`3Nf*J3vo^CRkeYl)Za>UoP>lgvADQIzQN{YjQEQ7kC=ny&dqT@ouswv2p}R;7xDFQL>$KQq>sX(pH4$x7rI4t(Q+K=N-qkEb~$-9wI5)8($_l!}5V$^#%q'
    'dC%L5Y%@j1+G3T}{sYv586RW->%-|hOJ~<F;j1WRXxUg7rN>`VO6gZZJ9Q^gU(h*)S$Zk&`Lm%TZrUs!4O2lbSDpmsdj3+5gL_l&{5+b)BpKPXqG2ve{TD;p'
    '4gSDM<%Kp*skH#bMvLiyS(fGiu?-G!Lbt*Jfb!~aA(CU_spmdFtPYx!IGPs77a|j_iG%)q<Riuc4oSt8<lLUdFkO&CW;IbZDZ7~AT}XN1kSV3THx;uDb8xtG'
    'zE5Tn^9Hj8^fXUNFI?yH<)`-K?6(fs3f+^bCp9#r#kDmFh0JSzV{3uCq~9+VMqfFhV(!WbFTEblcAYac(Om(jN=0qg1i@|n2R}Y}ZD(z|Xwi8RYC;@coO!*{'
    'kt)+IzY+(A$lp!KEA)m15?<8&8KlQh@Xx%SrDfDac^1ybi#!}ni|SvgaTE`-zNL9A_hwvc)t1{S=nxR*v$|Gm>K^5p#wtdP<7x|780|`)1pT-$5TwL&;77yF'
    's{o9RH%gb;Rei2vOgLmjsHMAdF%{FoYjuQ4kEP>A&Wm!lIM;hdzg2qnJ6t##0)%zrq%KriW)GzJj+H(f#)Z#loT6ZcEEv_rH`wgbx>r4lWmdC0kAnL}@kZFn'
    'ekz89k;U<Ni24vrG;NV7O^+HDd09eO4CLo7dB%LFYv6uK;3yG8iv{N;Q?aC0w&s<o#g)h5Yn<dP8pI#TB~kfR8=iTb!<wQk=Bf9uAH8{9FkhoSX?fmk_Bb%p'
    'fm1T@(<n~{X8Laq;@}QnVI3=DCr|=pl|-$(CLmvtd7NPV1Lad+?f@y}R%>+m8O57L!)RjVL&xV!n<F0;*g_j<6P80GO_Xk>)oia=9{jeLR91hg7lljG;c@iU'
    'v*lx(!v2T7aZ#0$U3S1!k$Vt#F1!|1y~*8rOwCU5=0RX|t41+GXFr{7KFL3%XPE)^&NDrrihM<d8A7~rqN+!pX2k&9le}N2+=af$TDsWk`L%atPojshN|7;3'
    'uC)EWq60+#jnjA@wO6C!Pp}xR%|30Kwc71exBJv;xpSA6<5^zOZadz1vfOnUh{`0HRt~GRv^EEEYz*T;bX`4y?KPaIovYbtE$^PRR~x9kaw9|M$qN7X@rJVC'
    '-H&pi`NP@wc4ztaSxh$2w;Ub%oHJiqdwz~SPxq;5h5^ZqYdmmoU6Xh$hsOJ4wn2UmOYgwYmOOUQwB0j3j0p1Y=m8(WZddK&4>;Rq3rtinj2dLMHC^QI6dJIG'
    '|Nb(5MTEW-(p`vp%b{70^U=%Aog6_CaqD<KiwAf;jqnW)@=5xh6^m#?S>BTrLtXPYR3a(|7MT-Cr@|d2UU6?pG#;my)Q!w8#h0a3;K_>XOb)Nm5jvTVueAV;'
    'vMrzwo-HD>Ou!`)@rxUXRseSttkUcYVDvBGEeKK*)Y7T-5OUayktdl^w1CY}d?fFaXviF>z`R`CbbU$8G$$~(A}=U4jCn7Lyjx6EX%^i=`M12iwouG*JF`%6'
    '8P%#|tY}=Z@P#?x*^zCQZ?s)Z@s?wwCF^G8(p6pp0=DPBIgT~6nDE`|T05dHO|Ms6<W-o6>30c`^lsEe;VL2?(U1HtZy4R0Ncs3ONa^P%TMhY1e(P;`H$Ea9'
    'aqKLY+!n!}<QTGoUcmB#B0vI^!bn$(m$S&nAV<k@ESZg56&d#&0XaOyi{V(l6^VvkzCa%=Q!gtTL4%^~FnIh}^<7)%#vWOusZ9K_*l6bMJ$XDY*@U?koBNA('
    '=&8Ggx|qlQ34l|=k}k~cSdC)Av2VMj35iB{rAfKZmiNdTO^aw1qS><KD1mQEQ3dI#;MxYle>9(G0TT#v*cREP#8}rtZqrRoOqS;8EB49PwmkeLI1}={(wv5`'
    'SqH(ehk~0n2F(uXJ;%tki*_<iCagQeIjYfA^jhdjcy^gk!gD8}j_VRx627aDNAsoX5+0%3JGqKcck+U6DD~r>*LnPSs{?zjQ(p_N&`ErT7xGoXSIW1ckZ*lM'
    'SfvFIrM#ejb<{87%*!cT_=pN6-)#GmVcM5@6e3rCw?!#{QD9jhSXvB4lm;Z*!;#Cc1ukI)f#p421~fzD)pDW9X!^)1hJsb}i7tjQ`KobhXu%%XJBcP&p1(qv'
    '(=;8gH47{l18J=szY4y`(P^C`{!$#Duehc=uab!PsJei%e6~=924-1-(~+1`hmECTvcyT4GgWrH;uyJUfex03D!J1O$}5x-UwUnIW9WFyUihc<6d`#^aHJY~'
    'k2E~{LC2hheGDo+kM!E}utKiRMQHt`)$rPN?UqVyn;VO~;PHr8ox*NZF6Q>M#D-J9tzBO=NC{!?)T2%tz<=k-81qe{pRk}=vN?~4ND;K$rAY(qOeuD+$I?qn'
    'd6AlOEWzCc8I&eMVu?lkT`>VcK-<e%wm_2Cx2Ta{1k`IQz?O!gR)Eg2N`xp1LkbfBFD~PF3SH==Gli-1mZkUHmDspdeHrDofi8=oOS~H!rH-q?c#-RLG_s>A'
    'I;PU}tkr$BxTKd{%WPi=*#$l@I6If(PL2kf`CSG`B@6=!thr5_?mo=O8^Oo9XUW#uR%(Gm_IJOr==QO!%wFJK;`L?a^hE~7NkT@C46m&vJ5W@_acT`zDOHhV'
    'EsEG^krpCHRsC41byq4R)4qdk3+oPhy($isZk^5X{?ElY&X)N@VH;hCu}8Qvkz2CcC$-_3kH?wruNl64cO-xGyA~+4mnkuGz(i|MEI>GsYy=#rcw`sOev``}'
    ')l^-hjyk4(Ec}lwk3E&KhKGCf)ie8cXZEPlF}FcM78qx}@=`VmP4EDE)^O}H=3&(<9ba}?IoF;E7hR;T6Wcf-OL9qfB^>aa{lNi2=XP&3D`3}xL$qg|+xLT<'
    '1WO*T$l#kVW+EQOaKpZ_L3t~8oWkt_{4BITfNeUB_ovesc!cTM%XdMlH_LH2iLQDE6#}+_Igp;oj@VgZbamCso$rWtc_Wq`43s0da4+D5S+O}9@hB*25h}t5'
    '3O@S{+cV#tYfn1hH`FQIu%id=o3G^W{u_ViumAmDfAk3de%B5)Dcv5+y6ROI#hT$K>SoywyEzt91&ou^wXb|#9^m0vlv?$=m#%1)2=GHhca$10;D&m?eqaYj'
    '!RS>~!`67C(6pfutGIj2G%KdA%%Q%u6jq;@L2HtgU=<a<8v9C6B26yI!%JgP<^7L}kCMpMU~u%t9OT53wV~ZpS~BzAJ^j|Sv{LL#@%~VD9=e={W|kR8ntt7+'
    '>uyn~D6qFF5@*OJStWdW4T67AuQ$)*t6_2$qtBp=VV9sRGt5=V=a$vJdm1{7Le7SyH`X1BWFgW|<>t0-IVDp*MbaTyX6=T)G}DBXUrz2IJEeZGH456ZIUjm_'
    'hingkN+Lfa$5sP5GIBmpuI2(aDakwcw+=~H_~s<$Ho5<ZwEee^|IvTr|IVXF`1hA>+HT#k!E{}R32LHF^xM2}L-YCL_yTqp=lO5eYRBgwY9uJ-<T3I{=pH<s'
    'XV)Ghig&CIfhV2_Fr&usa7eS{3@>xwA=Kn+7UI0clqIQQ)B9E%MyW0uE`&RKG`@_kbL5o0*_e*0ESlzkg_FEl`@kqidHk%GA<H_7jF`l!x(`Eph@(;Gd9AiS'
    '*YQJ!6y7^_@SaBuPL|dt)Td6dDKn?MY&Lke|N7bfH(%lPap9}|uU@}6{^r?X!!sXa(B_-BM{7Db^K{I-_hKj4#W5(dnG>dqXayPkavV={w9B!Fc=0LaI_+}w'
    'p<^(`xEOY5)>n=kT}CsCQKUW*zYuIwrB-AA$sC;o&{7)>PzmD=;MAJMDDB0OiiE+OAspZ#qv5u7<_`6Y#C`N07a<vxsSADzSNo-I+kC|brz60P|C58Sf6Byx'
    'hK$poX3((AWAUxLVzSFGz0Ml_Ok<1!h|r5Ef^9#)OvU_`<tw1&AWc7gD`*%h`{KeSB?iul8<uItaI%g6zr3ppwXCY+`;JcLpfIUvn&@%L@yzWyv%c5bBR8{1'
    '6hj5a2!`RbzV`Lfb?$V}o#x)iqM-0WBrzX?iio7AA|!|)s+a7chaO6ao&qC@9wMnEDD1!XUi*8WbM76L3TI{w=j^lBZ+-ng*7~pYU;oi0oDT@&11fag<&%Zj'
    '^|>ocaW-A&0X6bVPofvqdN50w^Tlc0aL%cyZX}{($BV<W<LU%sFvnE*XSc`2=jEcw(@$<~kMVIdx;+%t%2U)WW{O|t<+46_3{G}pj-s}KS*k);Mq$QMIAFB$'
    '><;EBJExmz2gf_wU@lh`dV8y-Cr*swA)ej-%e&Rfu~qfkJ0!dfIx*I_f`4Crk5GjluAOTZQf4cwT$NPii$!%ZhO-mZ^eePGJQzp%D30i?Mw%$T#WVWrJNc9<'
    '{$w|RX~vD8PM=&E!M|O6a+j<?f1TVRXR6mvhcf!Vms6LZL<4p)yyKm)5x9ecH|2OYeBKQW7*$kMv?~UTDHZbS=<efCrO6|k+vC!0sp*@LyMPr3abn}bm+JV-'
    '_SxJn(8b6rerE4TaUPcU$V%O`<+`3QIz=<fA44-^1==FLc4f~s-AZ&aS*aZb>#L$GD?>aBy*xv`w1obs5zr+*t7GuCT*=!;+}OyDf?Qb3V?9Uo`efQ{$Hd8n'
    'QYMr?v^^B}=3IVpo`e_YMObRSA&LwtbRPW-=$!Y*J(8MX$F+0<jno{(0q@C`Mtuh63bl`4A<7T!&+&baUXueJld)Fu!M;5t_v;-Q)n(5!a;EQltY7cM)?%l`'
    'Jvk&+^_g~SGvvJ9b6OwvSFdM7hlV^G3zgn*SD)dp9z9ZWkv!K|v%{iQ5#`vff#UXC!Q-of3~C}slQV7K$OiWe__$w;k6tceM)~q<$VcxzqiRD|TYG*>+ub1O'
    'ikim*(%}YDZ$nU@a=1cpgXw0rT?&Mjv;0gS`B}d4Wxw@Y<cGY`9-C+O@;lJn$wIfDaK1A?i`lT4G_ww!+;4_O7(6Imo;BYKP(xPDJh{yqa4T_TUEFTcDq+q!'
    '&At(^LRn}DoM8%_d%tL~Cp4V<dgj04ZllUn(b-sBY*>sw5#=P`70wuOmo!pls6ufA%e+NK-h;=R0kHBFNB{8~z5G~Qn+9ndYaH|#&3Y-+5bvRe%`A;$DEVQF'
    'GmX2~dsfzgNPP9n0}=T`+|T|pe>h7Om9LYjG8xp_TBEW$FjT!)n=M|Q*?+xk^G(4$lkz-+aId}dv8zGsmQuLCE{f&%R<=$L&|&v+MQgJT=(hvntyJIJ!urSn'
    'nao6-HTG}1;k=BoJi}}r?|N<qeQBA|+c${qciDBUw?uk9E4jr6!Xl1mPf9G#*bL-3Lvk$MR~WEK{*P@Kf9rcMzp!ydl7w_aXH%ggM9Q6oIr|A09tpPVPk4W>'
    'x|^!H=dfY<5Bm$gaMcg*#j7pjJ^?)2*a$my(`Lw-m8Kh2;ak<*H#k&XED-i4az$V5R*Z^br@8l4zC+P4#|OJ?wYRYEt%ZH=WY^mVpIR>>CB<K>&5cEy-Fa6{'
    '9y)bb2xL>tdG>Xy;ag;H0+?Z7o&oBF92%TTz%R0O!fb$X$Bhts?TaFpm&00gZCV@t9|TqCx5DjLO*)^R1F6wGdl^We9{W2=cj0PXwz1o^|CfsGNKkV{iF2>P'
    'ZSZB$gy2TBy|uY9Z8JetgYs&Yyt_;BTZGIgJ~u9*3g_1PWLHm9W_DI=L>UgI>bn?mMV}2Jzk=p(koxIiVjefYRFS{pIghGU;o_}^t5>y+82V@P(GP8Ho*PpV'
    '=*Z=6T&((-6ZYznEuRHx(+rl^L%C;jYyH#Exq9R7jrDWm*{{X{8J7{NFJ{xL4p%)$%fp4A+AQ|8X4LKevV+HA2v)0cScMdGRJXJj0~V)ar3XZA>dI-m|MR5s'
    'YR4kj>Kcot{&1C-A{D}Ie0K84CamyR@k;b}UtNITKHU!jI-esoX}Vm}uH;kZlJvW*bP5*URRocEAGmI+s=zws;y71?dM+oGr5<k*W@7}xt$E>+?0%25qs#7v'
    'IbZMTpM9<Bn{uXkea|$^6m!ddn4Wi6_VJ8%9yR6-SV+xTnEqE*q;nQ_jh0{R+N(UnG%w%6C0hsX&0nR3xQVTm%$y}q9#Eh`A_JZoZK(E58#(PZX<#wH&?+&<'
    '8JbA!3$0#U_V8+<z37heCYQ}$xdq)`k4O8q^8e2@%FXvaZ!cV)Bq7}jPrAohONWnEeC}`qTBO72i7ad@o+!<ilj)k9Y-}Mc`Q(B_2yy6y@(5La4cJMTY;n(x'
    '@EALftwkAS%3Fo7vL;Kdw8C>wR3<W>QpThd$f{K0EVD!-#R^Npa0LcVmQ66rcgm)Xy46)dl6rvVoLj^lvP@}?P%14>rK1MP)Wb)sG-WoE7Kw}@0nccPjbN;('
    'N*(sOym%g<GlE!#ImVDdAV(P{k|3oNW00SCBN6h5SdXDAgf&bUW{t5HYJ+j`4Gqx6O#Z|}AD-N`y&b|@ZJv$z+9@VHN}+PZ>lCY$ct@n++9GAGgSs0jGir6J'
    'wKmc+tPIk^d#^L8Q42p(UxfBT!ALNcF{&tYieL-<z=g?()M+X-cBwEX^Nbk`14J>0T2l?<rzMrP1zoSSBD%@2;#L`ADf574Q)g3?a;PB2(s*RC0Gbez($pC)'
    'pzhKr=pztc%vn5uUMt5~26f6PstAJ)$~2`!Q-L@YnJ@q@Qp_3S%6g?d<D9aLaFtnRpy#xpe5eKG>>-wbcuJTPnp#1$<|gAX&fE(LH8n<hW})*Xa+&oc#hPGh'
    'NGcQ(&LL;5oKr&UFnJgbNp(gj&uPyZO&!x%aGC`=V$ceYxt7c_YBLIjIGhP8Yz8&37<!^z)aIj)Ry0Hj7Qk=VInJ}xNUnjVr3hq2oikE!pjwbyf(e8Kp+YfA'
    '5cS4#!MSfYBF@655nFi<e4Sd&faatI`XDf(-T<vIiLFw?GeeBe6jwSEni&r)V3gFr2AJs<c59U!3xKu@!;SII5DKhCGU0$}o#z_H90?#^M-Z~mDTqr1LyVk;'
    ';qV6F;Yq82Ef3)Cyj?HsPi4<L?FF$QB&0S<V<1t0(4Jxi0)POClFVXH;T1!blu{Wji2_C@T4I^Dil{_W1m{42A_byW28d3A6tJl_8A1S;8ipj0BlU{VOi5#f'
    'V<4zfAZMh3c%hhK@GkKnpm9N<ud_@6#V|rr&m8q2)`+LlqYO%rHpmL>HNXS?66!GWDF&)=MJZ))R=u{i6>cE(leMjNH<|iy(_J|pID^__Ee!CQPJxRY2)@9)'
    '4zbiZg#}XD7>>9BzNZeDErni0!Yc!!wq4SoO!XXwlTu96RH4imB0!HgDJX*u!XV;!2J#I-%@L;Xg?NyJ0_YhthBWFF5!7lCv3JmoBiB@n5onW>D$~%u%p=3n'
    '%-M`*&>jgyW>Wz@3%p4sBHR%tIdm}A6gz@vmscGs4jo5R=qfKkMsNi35B_sHg$5Dm73_od0L}rW2^^^?mB69lu{S~(396%A%252=AaywqlMLz=WUr7~LJ^b#'
    'ZF!0iw(x<Z)&eI{sxZhk22#c_mD*G>4$`zJq!V8y(3j-KIuM15!W4~5&>AT>+zC)fIs;0x&chHX7zyQ#gMo8ALzZWtNF1<D54S%Ggb@%Xo@dq}pe%wN6HbG0'
    'F<3fm5s1|cLFb7uP7=5(biXyJkPLIh(q0km?g#w^jK>@|G^HS|DWRa|wF(*m12cnO1uj=W!AMDHO-ez!@KigBC=`hL@Ez-Jm<%w`(gFjk%UnWR#R+7E%0k0H'
    '`KgSP)Ttm#ECDS9<OPC;7-5`n8Kp-KJq9zkr32SoQqSE!{@Sm<a@A6jgd~&t&$mAx{=dliS+|g<Z+h~U!%#p-ceM&xTbfbGe&%4^;{JH~Z?F0hd<y9ut>Ruh'
    'kR&~s(9Qmjk9_IggYQj}kgjQE|H&muGBhFSB7F0SFMr^{>yjj-8(Ii|zpaDNnsjt=e)F&UkNzG;C8T$?a2~s}gVUH;bWuL@o$dRGN|KOnX`wuQVFpT*dAe{f'
    'KYzz_^yVZ9>2M3~((l)B<piOd`?F7--M#dkNfOdkt=#@sRqn<lpqu}r*FFBld#+59kgjXx|LPyA{9Q%g1@Yhg_ZOZ6QV8k#7Ra@)R3Lq2-i3MQ?RT7e_|hZ^'
    '>Dm^|Yk#X?I+N}$#xH;V?juiKog^W(Y4!1cRv7ifx|{v*4{m+(#T%0(q^n!m@40d*8JIJ40S<ob@*jWwP?CgnV+-Jc<vzeXx>&yR&p&<jiX;hX)IvJSmy$)g'
    'zl(PC7oU6eC6pu~wTI~R2Wzy2y!^XY4!*DhtP|4F7R>o+4KrVdcQJ2&@*6Kb3wR;5y9_@uj9Ds*E>bsoeT{VZ*`;~0?7KkS2xc|V!#`Z=#53zramOXK&ik`J'
    'EzNdN?ZR}upK6$|{bgy`7pe=<-RrC&{(9MgIXj<SoUT2h#(CzN1ADVhJb2&1w}mX%!{6I3OOgj}0#T9t2YfbXqW'
)
EMBEDDED_SOURCE_SHA256 = '69ad25326188513d3b1b4928015f7be48ef0deb294b128ddf7d44796fa4f1c69'
PROJECT = WORKING_ROOT / 'TickNets_CIFAR10_LLargeV2'

def project_prefixes(archive):
    names = {info.filename.replace('\\', '/') for info in archive.infolist() if not info.is_dir()}
    prefixes = []
    for name in names:
        if name == 'train_cifar10_large_v2.py' or name.endswith('/train_cifar10_large_v2.py'):
            prefix = name[:-len('train_cifar10_large_v2.py')]
            if all(prefix + relative in names for relative in SOURCE_PATHS):
                prefixes.append(prefix)
    return sorted(set(prefixes))

def find_project_source():
    if SOURCE_ZIP is not None and SOURCE_ROOT is not None:
        raise ValueError('Set only one of SOURCE_ZIP or SOURCE_ROOT')
    if SOURCE_ZIP is not None:
        candidate = Path(SOURCE_ZIP)
        if not candidate.is_file():
            raise FileNotFoundError(f'Source ZIP not found: {candidate}')
        with zipfile.ZipFile(candidate) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError(f'Source ZIP must contain one complete project; found {prefixes}')
        return 'zip', candidate, prefixes[0]
    if SOURCE_ROOT is not None:
        candidate = Path(SOURCE_ROOT)
        if not candidate.is_dir() or not all((candidate / relative).is_file() for relative in SOURCE_PATHS):
            raise FileNotFoundError(f'Source folder is incomplete: {candidate}')
        return 'directory', candidate, ''
    zip_candidates = []
    directory_candidates = []
    visited = set()
    for current, directories, files in os.walk(KAGGLE_INPUT, followlinks=True):
        resolved = Path(current).resolve()
        if resolved in visited:
            directories[:] = []
            continue
        visited.add(resolved)
        if 'train_cifar10_large_v2.py' in files and all(
                (Path(current) / relative).is_file() for relative in SOURCE_PATHS):
            directory_candidates.append(Path(current))
            directories[:] = []
            continue
        for filename in files:
            if not filename.lower().endswith('.zip'):
                continue
            candidate = Path(current) / filename
            try:
                with zipfile.ZipFile(candidate) as archive:
                    for prefix in project_prefixes(archive):
                        zip_candidates.append((candidate, prefix))
            except zipfile.BadZipFile:
                continue
    if len(zip_candidates) == 1:
        candidate, prefix = zip_candidates[0]
        return 'zip', candidate, prefix
    if not zip_candidates and len(directory_candidates) == 1:
        return 'directory', directory_candidates[0], ''
    if not zip_candidates and not directory_candidates:
        import base64, zlib
        payload = zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85))
        if hashlib.sha256(payload).hexdigest() != EMBEDDED_SOURCE_SHA256:
            raise ValueError('Embedded Phase 14 source checksum differs')
        embedded_path = WORKING_ROOT / 'TickNets_CIFAR10_LLargeV2_Embedded_Source.zip'
        embedded_path.parent.mkdir(parents=True, exist_ok=True)
        embedded_path.write_bytes(payload)
        with zipfile.ZipFile(embedded_path) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError('Embedded Phase 14 source is incomplete')
        return 'zip', embedded_path, prefixes[0]
    attached = sorted(path.name for path in Path(KAGGLE_INPUT).iterdir()) if Path(KAGGLE_INPUT).is_dir() else []
    raise RuntimeError('Could not identify one complete TickNets source in Kaggle Input. '
                       f'ZIP matches: {zip_candidates}; folder matches: {directory_candidates}; '
                       f'Input entries: {attached}. Attach the source dataset, or set '
                       'SOURCE_ZIP / SOURCE_ROOT to its exact path.')

source_type, source_path, prefix = find_project_source()
if source_type == 'zip':
    with zipfile.ZipFile(source_path) as archive:
        members = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            normalized = info.filename.replace('\\', '/')
            if normalized in members:
                raise ValueError('Source ZIP contains duplicate filenames')
            members[normalized] = info.filename
        manifest_name = prefix + 'cifar10_large_v2_source_manifest.json'
        manifest = json.loads(archive.read(members[manifest_name])) if manifest_name in members else None
        payloads = {relative: archive.read(members[prefix + relative]) for relative in SOURCE_PATHS}
else:
    manifest_path = source_path / 'cifar10_large_v2_source_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else None
    payloads = {relative: (source_path / relative).read_bytes() for relative in SOURCE_PATHS}
if manifest is not None and set(manifest) != set(SOURCE_PATHS):
    raise ValueError('Source manifest does not match the required files')
PROJECT.mkdir(parents=True, exist_ok=True)
for relative, payload in payloads.items():
    parts = PurePosixPath(relative).parts
    if not parts or any(part in ('', '.', '..') for part in parts):
        raise ValueError(f'Unsafe source path: {relative}')
    if manifest is not None and hashlib.sha256(payload).hexdigest() != manifest[relative]:
        raise ValueError(f'Source checksum differs: {relative}')
    target = (PROJECT / Path(*parts)).resolve()
    if not target.is_relative_to(PROJECT.resolve()):
        raise ValueError(f'Source path escapes project: {relative}')
    if target.exists() and target.read_bytes() != payload:
        raise RuntimeError(f'Existing extracted source differs: {target}; use a fresh session')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
missing = [module for module in ('torch', 'torchvision', 'numpy', 'PIL', 'pandas')
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f'Missing Kaggle packages: {missing}')
import torch, torchvision
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running.'
print('Project source:', source_type, source_path)
print('CIFAR-10 Large v2 source:', hashlib.sha256((PROJECT / 'train_cifar10_large_v2.py').read_bytes()).hexdigest())
print('Runtime:', torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
import subprocess
command = [sys.executable, 'scripts/run_kaggle_cifar10_large_v2.py',
           '--cifar10-source-root', str(CIFAR10_SOURCE_ROOT)]
if EPOCHS_PER_SESSION is not None:
    command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
if RESUME_ROOT is not None:
    command += ['--resume-root', str(RESUME_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)
